## Phase 0: Configuration and Field Mapping

Issues:
* LOINC Answers aren't being assigned to their LOINC List in hierarchy

Known missing pieces:
* Community Mappings
* be sure to include mapping guidance. Examples:
        https://loinc.org/60567-5/ Term Description – this is “Comment” in the 2.71 repo version, and Joe is now modeling as a “Description” for 2.80 and 2.81
        https://loinc.org/74774-1 - Reference Information -> Mapping Guidance – this term should not be used in the US at all
* Richer hierarchy, more like 2.71

In [ ]:
### LOINC FILE DISCOVERY AND COMPILATION ###

import os
import shutil
import re
from pathlib import Path

# Set the source directory where LOINC files are located (in various subfolders)
source_loinc_directory = r"C:/Users/jamlung/Documents/LOINC/Loinc_2.81"  # UPDATE THIS PATH
# source_loinc_directory = r"c:/Users/jamlung/Downloads/SmallTestCSVs" # For testing with small set of files

# Define the target files we need to find and their expected names
required_files = {
    'Loinc.csv': ['Loinc.csv', 'LOINC.csv', 'loinc.csv'],
    'Part.csv': ['Part.csv', 'PartLink.csv', 'PART.csv', 'part.csv'],
    'AnswerList.csv': ['AnswerList.csv', 'ANSWERLIST.csv', 'answerlist.csv'],
    'PanelsAndForms.csv': ['PanelsAndForms.csv', 'PANELSANDFORMS.csv', 'panelsandforms.csv'],
    'LoincAnswerListLink.csv': ['LoincAnswerListLink.csv', 'LoincAnswerListLink.csv', 'loincanswer listlink.csv'],
    'MapTo.csv': ['MapTo.csv', 'MAPTO.csv', 'mapto.csv'],
    'ComponentHierarchyBySystem.csv': ['ComponentHierarchyBySystem.csv', 'COMPONENTHIERARCHYBYSYSTEM.csv', 'componenthierarchybysystem.csv']
}

def is_linguistic_variant_file(filename, filepath):
    """
    Determines if a file is likely a linguistic variant based on flexible patterns.
    """
    # Skip known main LOINC files
    main_file_patterns = [
        'loinc.csv', 'part.csv', 'answerlist.csv', 'panelsandforms.csv',
        'loincanswer', 'mapto.csv', 'componenthierarchy'
    ]

    filename_lower = filename.lower()
    if any(pattern in filename_lower for pattern in main_file_patterns):
        return False

    # Pattern 1: Files starting with 2-letter language code (xx.csv or xxYY.csv)
    pattern1 = r'^[a-z]{2}([A-Z]{2})?\.csv$'
    if re.match(pattern1, filename):
        return True

    # Pattern 2: Files starting with language-country code (e.g., esMX_something.csv)
    pattern2 = r'^[a-z]{2}[A-Z]{2}[_-].*\.csv$'
    if re.match(pattern2, filename):
        return True

    # Pattern 3: Files in directories suggesting linguistic variants
    path_indicators = ['linguistic', 'variant', 'translation', 'locale', 'lang', 'i18n', 'l10n']
    if any(indicator in filepath.lower() for indicator in path_indicators):
        return True

    # Pattern 4: Files with linguistic indicators in name
    name_indicators = ['linguistic', 'variant', 'translation', 'locale']
    if any(indicator in filename_lower for indicator in name_indicators):
        return True

    # Pattern 5: More flexible language code detection
    # Check if first 2-4 characters look like language codes
    if len(filename) >= 5:  # Minimum: xx.csv (5 chars)
        # Check for 2-letter language code followed by delimiter
        if (filename[:2].islower() and filename[:2].isalpha() and
            filename[2] in ['.', '_', '-']):
            return True

        # Check for 4-letter language-country code followed by delimiter
        if (len(filename) >= 7 and  # Minimum: xxYY.csv (7 chars)
            filename[:2].islower() and filename[:2].isalpha() and
            filename[2:4].isupper() and filename[2:4].isalpha() and
            filename[4] in ['.', '_', '-']):
            return True

    return False

def find_and_compile_loinc_files(source_dir, target_dir='Input'):
    """
    Searches for LOINC CSV files in subdirectories and copies them to a target folder.
    Also handles linguistic variant files specially.
    """
    # Create target directory
    os.makedirs(target_dir, exist_ok=True)

    print(f"🔍 Searching for LOINC files in: {source_dir}")
    print(f"📁 Target directory: {target_dir}")

    found_files = {}
    missing_files = []

    # Search for each required file
    for target_name, possible_names in required_files.items():
        file_found = False

        # Walk through all subdirectories
        for root, dirs, files in os.walk(source_dir):
            for filename in files:
                if filename in possible_names:
                    source_path = os.path.join(root, filename)
                    target_path = os.path.join(target_dir, target_name)

                    # Copy file to target directory
                    shutil.copy2(source_path, target_path)
                    found_files[target_name] = source_path
                    print(f"✅ Found and copied: {filename} → {target_name}")
                    file_found = True
                    break

            if file_found:
                break

        if not file_found:
            missing_files.append(target_name)
            print(f"❌ Not found: {target_name} (searched for: {', '.join(possible_names)})")

    # Handle linguistic variant files specially - preserve original names
    linguistic_variants_dir = os.path.join(target_dir, 'LinguisticVariants')
    os.makedirs(linguistic_variants_dir, exist_ok=True)

    linguistic_files_found = 0
    print(f"\n🌐 Searching for linguistic variant files (preserving original names)...")

    for root, dirs, files in os.walk(source_dir):
        for filename in files:
            if filename.endswith('.csv'):
                full_path = os.path.join(root, filename)

                if is_linguistic_variant_file(filename, full_path):
                    source_path = full_path
                    # Preserve original filename - no renaming
                    target_path = os.path.join(linguistic_variants_dir, filename)

                    shutil.copy2(source_path, target_path)
                    linguistic_files_found += 1
                    print(f"🌐 Found linguistic variant: {filename} (from {os.path.relpath(root, source_dir)})")

    print(f"\n📊 Summary:")
    print(f"   Found {len(found_files)}/{len(required_files)} required files")
    print(f"   Found {linguistic_files_found} linguistic variant files")

    if missing_files:
        print(f"   Missing files: {', '.join(missing_files)}")
        print(f"   ⚠️  You may need to manually locate and copy these files")

    return found_files, missing_files, linguistic_files_found

def update_file_paths(target_dir='Input'):
    """
    Updates the file path variables to point to the compiled Input directory.
    """
    global loinc_csv_path, part_link_csv_path, answer_list_csv_path
    global linguistic_variants_path, panels_and_forms_csv_path
    global loinc_answer_list_link_csv_path, map_to_csv_path, component_hierarchy_file_path

    # Update all file paths to point to Input directory
    loinc_csv_path = os.path.join(target_dir, 'Loinc.csv')
    part_link_csv_path = os.path.join(target_dir, 'Part.csv')
    answer_list_csv_path = os.path.join(target_dir, 'AnswerList.csv')
    linguistic_variants_path = os.path.join(target_dir, 'LinguisticVariants')
    panels_and_forms_csv_path = os.path.join(target_dir, 'PanelsAndForms.csv')
    loinc_answer_list_link_csv_path = os.path.join(target_dir, 'LoincAnswerListLink.csv')
    map_to_csv_path = os.path.join(target_dir, 'MapTo.csv')
    component_hierarchy_file_path = os.path.join(target_dir, 'ComponentHierarchyBySystem.csv')

    print(f"\n🔄 Updated file paths to use {target_dir} directory:")
    print(f"   LOINC CSV: {loinc_csv_path}")
    print(f"   Part CSV: {part_link_csv_path}")
    print(f"   Answer List CSV: {answer_list_csv_path}")
    print(f"   Linguistic Variants: {linguistic_variants_path}")
    print(f"   Panels and Forms CSV: {panels_and_forms_csv_path}")
    print(f"   LOINC Answer List Link CSV: {loinc_answer_list_link_csv_path}")
    print(f"   Map To CSV: {map_to_csv_path}")
    print(f"   Component Hierarchy CSV: {component_hierarchy_file_path}")

# Check if source directory exists and run the compilation
if os.path.exists(source_loinc_directory):
    print("🚀 Starting LOINC file discovery and compilation...")
    found, missing, linguistic_count = find_and_compile_loinc_files(source_loinc_directory)

    if len(found) >= 4:  # Minimum files needed to proceed
        update_file_paths()
        print("\n✅ File compilation completed successfully!")
        print("   You can now proceed with the rest of the notebook.")
    else:
        print(f"\n❌ Compilation incomplete. Found only {len(found)} out of {len(required_files)} required files.")
        print("   Please check your source directory path and ensure LOINC files are present.")

else:
    print(f"❌ Source directory not found: {source_loinc_directory}")
    print("   Please update the 'source_loinc_directory' variable with the correct path.")
    print("   If you don't have LOINC files, the notebook will use demo/test data instead.")

In [ ]:
### CONFIGURATION & DEMO VARIABLES ###

# Specify the organization and source to output
global_organization = "Regenstrief"
global_source = "LOINC-2-81"

# Set the path for your input CSV files
loinc_csv_path = 'Input/Loinc.csv'
part_link_csv_path = 'Input/Part.csv'
answer_list_csv_path = 'Input/AnswerList.csv'
linguistic_variants_path = 'Input/LinguisticVariants'
panels_and_forms_csv_path = 'Input/PanelsAndForms.csv'
loinc_answer_list_link_csv_path = 'Input/LoincAnswerListLink.csv'
map_to_csv_path = 'Input/MapTo.csv'
component_hierarchy_file_path = "Input/ComponentHierarchyBySystem.csv"
config_path = 'UMLS_API_config.json'


# Set the output path for the transformed JSON files
output_folder = 'output'

# Set this to 1 or 2 to run the notebook with example data instead of your real CSVs.
# NOTE: The demo data below is for illustration; real-world data is much larger.
mode = 0  # 0 = full run with all data, 1 = test_mode with real subset of data up to 15 records per CSV, 2 = demo_mode with example data

In [ ]:
### PACKAGE IMPORTS ###

# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations, often used for NaN values
import numpy as np

# Import json for saving to JSON Lines format
import json

#Import StringIO to handle in-memory text streams
from io import StringIO

# Import os for file and directory operations
import os

# Import re for regular expression operations
import re

# Import time for timing operations
import time

# For ignoring performance warnings from pandas
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

from typing import List, Dict, Set

import ast # Used for safely evaluating string-formatted dictionaries

import requests

import sys

In [ ]:
# Example DataFrames for demo_mode (mode = 2)
# These are based on the CSV snippets you provided.
demo_loinc_df = pd.DataFrame({
    'LOINC_NUM': ['100000-9', '100001-7'],
    'COMPONENT': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'PROPERTY': ['Hx', 'LP431396-3'],
    'TIME_ASPCT': ['Pt', 'Pt'],
    'SYSTEM': ['^Patient', 'Ser'],
    'SCALE_TYP': ['Nar', 'Qn'],
    'SHORTNAME': ['Health Info Pioneer+Father of LOINC', 'Health Info Pioneer+Cofounder of LOINC'],
    'LONG_COMMON_NAME': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'STATUS': ['ACTIVE', 'ACTIVE']
})

demo_part_link_df = pd.DataFrame({
    'LoincNumber': ['100000-9', '100000-9', '100000-9', '100000-9', '100000-9'],
    'LongCommonName': ['Health informatics pioneer and the father of LOINC'] * 5,
    'PartNumber': ['LP431397-1', 'LP6817-3', 'LP6960-1', 'LP310005-6', 'LP7749-7'],
    'PartName': ['Health informatics pioneer and the father of LOINC', 'Hx', 'Pt', '^Patient', 'Nar'],
    'PartCodeSystem': ['http://loinc.org'] * 5,
    'PartTypeName': ['COMPONENT', 'PROPERTY', 'TIME', 'SYSTEM', 'SCALE'],
    'LinkTypeName': ['Primary'] * 5,
    'Property': ['http://loinc.org/property/COMPONENT', 'http://loinc.org/property/PROPERTY', 'http://loinc.org/property/TIME_ASPCT', 'http://loinc.org/property/SYSTEM', 'http://loinc.org/property/SCALE_TYP']
})

demo_answer_list_df = pd.DataFrame({
    'AnswerListId': ['LL1000-0', 'LL1000-0', 'LL1000-0', 'LL1001-8'],
    'AnswerListName': ['PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_14_30D freq amts'],
    'AnswerListOID': ['1.3.6.1.4.1.12009.10.1.165'] * 3 + ['1.3.6.1.4.1.12009.10.1.166'],
    'ExtDefinedYN': ['N'] * 4,
    'AnswerStringId': ['LA13825-7', 'LA13838-0', 'LA13892-7', 'LA6270-8'],
    'SequenceNumber': [1, 2, 3, 1],
    'DisplayText': ['1 slice or 1 dinner roll', '2 slices or 2 dinner rolls', 'More than 2 slices or 2 dinner rolls', 'Never']
})

# Linguistic variant CSV demo data
esMX_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
33512-5,Color,Tipo,Punto temporal,XXX,Nominal,,,,Color: XXX : Punto temporal: Tipo: Nominal:,,
24355-0,Panel macroscópico de análisis de orina,-,Punto temporal,Orina,-,,,,Panel macroscópico de análisis de orina: Orina : Punto temporal: -: -:,,
10003-2,Duración de la onda R. derivación III,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación III:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10006-5,Duración de la onda R. derivación V3,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V3:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10007-3,Duración de la onda R. derivación V4,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V4:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10024-8,Duración de la onda R 'plomo AVR,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R 'plomo AVR:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
1004-1,"Test de antiglobulina directo, reactivo específico del complemento",Presencia o umbral,Punto temporal,Eritrocitos,Ordinal,,,,"Test de antiglobulina directo, reactivo específico del complemento: Eritrocitos : Punto temporal: Presencia o umbral: Ordinal:",,
10060-2,Amplitud de onda S. Conduzca AVR,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. Conduzca AVR:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10063-6,Amplitud de onda S. derivación III,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. derivación III:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10280-6,Número de modelo del proveedor,Tipo,Punto temporal,Tubo de cobre,Nominal,,,,Número de modelo del proveedor:Tubo de cobre :Punto temporal:Tipo:Nominal:,,
10445-5,CD11c Ag,Presencia o umbral,Punto temporal,Tejido y frotis,Ordinal,Mancha inmune,,,CD11c Ag: Tejido y frotis : Punto temporal: Presencia o umbral: Ordinal: Mancha inmune,,
10455-4,Xilosa ^30 M después de 25 g de xilosa VO,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Xilosa : Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10539-5,glipizida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,glipizida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10547-8,Primidona + FENobarbital,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Primidona + FENobarbital: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10550-2,Temazepam,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Temazepam: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
1099-1,K sub p super sub a Ab,Presencia o umbral,Punto temporal,Suero o Plasma,Ordinal,,,,K sub p super sub a Ab: Suero o Plasma : Punto temporal: Presencia o umbral: Ordinal:,,
10995-9,Neomicina,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Neomicina: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
11001-5,Pirazinamida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Pirazinamida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,"""

etEE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
93488-5,Guanidinoatsetaat,SCnc,Pt,Vereplekk,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Veri,
93505-6,Heptakarboksüülporfüriin I,SRat,24 tunni,U,Qn,,CHEM,,,Aine määr Kvantitatiivne Uriin,
93729-2,Beeta-2-mikroglobuliin/kreatiniin,Suhe,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
93748-2,Fibriini monomeerid,MCnc,Pt,PPP,Qn,IA,COAG,,,Juhuslik Kvantitatiivne Trombotsüütidevaene plasma,
95073-3,Histoplasma capsulatum antigeen,MCnc,Pt,BalF,Qn,IA,MICRO,,,Juhuslik Kvantitatiivne,
95074-1,Bakterid,PrThr,Pt,BalF,Ord,Valgusmikroskoopia,MICRO,,,Järgarvuline Juhuslik,
89481-6,Gentamütsiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92255-9,Metitsilliin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92242-7,Pürasiinamiid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96635-8,HLA-C,Tüüp,Pt,B/Tis^doonor,Nom,,HLA,,,Juhuslik Kude Veri Veri või koematerjal,
95563-3,16-alfahüdroksüdehüdroepiandrosteroon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95593-0,25-hüdroksükaltsiferool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95114-5,Insuliin^2 tundi pärast sööki,Acnc,Pt,S/P,Qn,,CHAL,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
93773-0,11-deoksükortisool,SCnc,Pt,Sal,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Sülg,
93838-1,Histoplasma capsulatum antikehad.IgM,Acnc,Pt,CSF,Qn,IA,MICRO,,,Immuunglobuliin M Juhuslik Kvantitatiivne Liikvor,
94255-7,Kaltsium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94256-5,Magneesium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94270-6,Ubikinoon 10,SCnt,Pt,WBC,Qn,,CHEM,,,Ainehulga sisaldus Juhuslik Kvantitatiivne Leukotsüüdid,
95543-5,Aspergillus terreus antikehad.IgG,PrThr,Pt,S,Ord,,ALLERGY,,,Immuunglobuliin G Järgarvuline Juhuslik Seerum,
95527-8,Tsütomegaloviirus antikehad.IgG,PrThr,Pt,Sal,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Sülg,
95688-8,Dengue viiruse 1.+ 2.+ 3.+ 4. tüüp antikehad.IgM,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin M Järgarvuline Juhuslik Täpsustamata materjal,
93771-4,Kalprotektiin,MCnc,Pt,SynF,Qn,,CHEM,,,Juhuslik Kvantitatiivne Liigesevedelik sünoviaalvedelik,
95574-0,17-alfahüdroksüpregnanoloon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95594-8,Kaltsidiool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95675-5,Kollapalaviku viirus antikehad.IgG,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Täpsustamata materjal,
95719-1,Flaviviirus antikehad,PrThr,Pt,S/P,Ord,,MICRO,,,Järgarvuline Juhuslik Plasma Seerum Seerum või plasma,
95800-9,Immuunglobuliini vabad kerged ahelad.paneel,-,-,U,-,,PANEL.CHEM,,,Uriin,
95966-8,Aspergillus glaucus antikehad.IgE,Acnc,Pt,S,Qn,,ALLERGY,,,Immuunglobuliin E Juhuslik Kvantitatiivne Seerum,
96043-5,Uratsüül,MCnc,Pt,S/P,Qn,,CHEM,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
96108-6,Klofasimiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96111-0,Linetsoliid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,"""

frBE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
103631-8,Natalizumab,Concentration de masse,Temps ponctuel,Sérum,Ordinal,IA,Médicaments et produits toxiques,,,,
106016-9,Bactéries,Présence ou identité,Temps ponctuel,Pénis,Nominal,Culture,Microbiologie,,,Verge,
106033-4,Bactéries,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture anaérobique,Microbiologie,,,,
106034-2,Champignon,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture,Microbiologie,,,,
103648-2,Hormone folliculo-stimulante^4 h post dose hormone de libération des gonadotrophines,Concentration arbitraire,Temps ponctuel,Sérum/Plasma,Quantitatif,,Tests de provocation,,,"4 h post dose GNRH FSH Gn-RF, Gonadotrophines-releasing factor",
103685-4,Citalopram,Concentration de masse,Temps ponctuel,Urine,Quantitatif,LC/MS/MS,Médicaments et produits toxiques,,,,
103806-6,Note,Observation,Temps ponctuel,Contact téléphonique,Document,Oncologie,DOC.CLINRPT,,,,
103830-6,Gabapentine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103834-8,Norbuprenorphine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103839-7,Phentermine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103958-5,Ofloxacine,Susceptibilité,Temps ponctuel,Isolat,Ordinal,Génotypage,Sensibilité aux antibiotiques,,,,
104133-4,Éthanol,PrThr,Temps ponctuel,Gaz expiré,Ordinal,,Médicaments et produits toxiques,,,,
104181-3,Toxine du clostridium tetani,PrThr,Temps ponctuel,Sérum/Plasma,Ordinal,Test biologique sur souris,Microbiologie,,,,
104183-9,Adénovirus ADN,PrThr,Temps ponctuel,Spécimen conjonctival,Ordinal,Sonde avec amplification de la cible,Microbiologie,,,,
104196-1,Créatine/Créatinine,Ratio de substance,Temps ponctuel,Sang sur papier filtre,Quantitatif,,Chimie,,,,
104234-0,Atomoxétine,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104237-3,Zopiclone,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104419-7,Legionella sp Ac^1er échantillon,Titre,Temps ponctuel,Sérum,Ordinal,IA,Microbiologie,,,Anticorps Echantillon.1,
104457-7,Virus varicelle-zona Anticorps.IgA,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,
104459-3,Virus varicelle-zona Anticorps.IgG,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,"""


In [ ]:
### FIELD MAPPING DATASET - LOINCs ###

# This dictionary maps LOINC CSV fields to their corresponding OCL Concept fields.
# Mappings are based on the provided PDF and the LOINC FHIR example.
loinc_to_ocl_mapping = {
    # General Format: '[Loinc Field]':'[OCL field]'

    #Field Mappings
    'LOINC_NUM': ['id','extras.Code_in_Source'],
    'LONG_COMMON_NAME': ['names.Long-Common.en-GB[1]','extras.LONG_COMMON_NAME'],
    'DisplayName': ['names.Display.en-GB[1]','extras.DisplayName'],
    'SHORTNAME': ['names.Short.en-GB[1]','extras.SHORTNAME'],
    'CONSUMER_NAME': ['names.Consumer.en-GB[1]','extras.CONSUMER_NAME'],
    'SCALE_TYP': ['datatype','extras.SCALE_TYP'],
    'STATUS': ['retired','extras.STATUS'], # Retired translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
    'DefinitionDescription': ['description','extras.DEFINITION_DESCRIPTION'],

    # Mappings derived from LOINC FHIR example JSON
    # These will be stored as `extras` to align with the FHIR properties.
    'COMPONENT': 'extras.COMPONENT',
    'PROPERTY': 'extras.PROPERTY',
    'TIME_ASPCT': 'extras.TIME_ASPCT',
    'SYSTEM': 'extras.SYSTEM',
    'METHOD_TYP': 'extras.METHOD_TYP',
    'VersionFirstReleased':'extras.VersionFirstReleased',
    'VersionLastChanged':'extras.VersionLastChanged',
    'ORDER_OBS':'extras.ORDER_OBS',
    'HL7_FIELD_SUBFIELD_ID':'extras.HL7_FIELD_SUBFIELD_ID',
    'EXTERNAL_COPYRIGHT_NOTICE':'extras.EXTERNAL_COPYRIGHT_NOTICE',
    'SURVEY_QUEST_TEXT':'extras.SURVEY_QUEST_TEXT',
    'SURVEY_QUEST_SRC':'extras.SURVEY_QUEST_SRC',
    'UNITSREQUIRED':'extras.UNITSREQUIRED',
    'RELATEDNAMES2':'extras.RELATEDNAMES2',
    'EXTERNAL_COPYRIGHT_LINK': 'extras.EXTERNAL_COPYRIGHT_LINK',
    'ValidHL7AttachmentRequest': 'extras.ValidHL7AttachmentRequest',
    'CHNG_TYPE': 'extras.CHNG_TYPE',
    'STATUS_TEXT': 'extras.STATUS_TEXT',
    'STATUS_REASON': 'extras.STATUS_REASON',
    'PanelType': 'extras.PanelType',
    'CHANGE_REASON_PUBLIC': 'extras.CHANGE_REASON_PUBLIC',
    'COMMON_TEST_RANK': 'extras.COMMON_TEST_RANK',
    'AskAtOrderEntry': 'extras.AskAtOrderEntry',
    'AssociatedObservations': 'extras.AssociatedObservations',
    'EXAMPLE_UNITS': 'extras.EXAMPLE_UNITS',
    'EXMPL_ANSWERS': 'extras.EXMPL_ANSWERS',
    'EXAMPLE_UCUM_UNITS': 'extras.EXAMPLE_UCUM_UNITS',
    'HL7_ATTACHMENT_STRUCTURE': 'extras.HL7_ATTACHMENT_STRUCTURE',
    'COMMON_ORDER_RANK': 'extras.COMMON_ORDER_RANK',
    'FORMULA': 'extras.FORMULA',
    'CLASS': 'extras.CLASS',
    'CLASSTYPE': 'extras.CLASSTYPE'

}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc = {
    'type': 'Concept',
    'concept_class': 'LOINC',
    'source': global_source,
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'extras.Code_Type': 'LOINC'
}

In [ ]:
### FIELD MAPPING DATASET - LOINC Parts ###

loinc_part_to_ocl_mapping ={
    "PartNumber": ["id", "extras.Code_in_Source"],
    "PartTypeName": "extras.PartTypeName",
    "PartName": ["names.Fully-Specified.en-GB[1]", "extras.LONG_COMMON_NAME"],
    "PartDisplayName": ["names.Display.en-GB[1]", "extras.PartDisplayName"],
    'Status': ['retired','extras.STATUS'] # 'retired' translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc_parts = {
    'type': 'Concept',
    'concept_class': 'LOINC Part',
    'datatype': 'N/A',
    'source': global_source,
    'owner_type': 'Organization',
    'owner': global_organization,
    'extras.Code_Type': 'LOINC Part'
}

In [ ]:
### FIELD MAPPING DATASET - LOINC Answer Lists and Answers ###

# Answer Lists
answer_list_to_ocl_mapping = {
    "AnswerListId": ["id", "extras.Code_in_Source"],
    "AnswerListName": ["names.Fully-Specified.en-GB[1]", "extras.AnswerListName"],
    "AnswerListOID": "extras.AnswerListOID",
    "ExtDefinedYN": "extras.ExtDefinedYN",
    "ExtDefinedAnswerListCodeSystem": "extras.ExtDefinedAnswerListCodeSystem",
    "ExtDefinedAnswerListLink": "extras.ExtDefinedAnswerListLink"
}

fixed_values_answer_list = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "Answer List",
    "datatype": "N/A",
    "source": global_source,
    "owner_type": "Organization",
    "owner": global_organization,
    "extras.Code_Type": "LOINC Answer List"
}

# Answers
answer_to_ocl_mapping = {
    "AnswerStringId": ["id", "extras.Code_in_Source"],
    "DisplayText": ["names.Display.en-GB[1]", "extras.DisplayText"],
    "LocalAnswerCode": "extras.LocalAnswerCode",
    "LocalAnswerCodeSystem": "extras.LocalAnswerCodeSystem",
    "SequenceNumber": "extras.SequenceNumber",
    "ExtCodeId": "extras.ExtCodeId",
    "ExtCodeDisplayName": ["names.Fully-Specified.en-GB[1]", "extras.ExtCodeDisplayName"],
    "ExtCodeSystem": "extras.ExtCodeSystem",
    "ExtCodeSystemVersion": "extras.ExtCodeSystemVersion",
    "ExtCodeSystemCopyrightNotice": "extras.ExtCodeSystemCopyrightNotice",
    "SubsequentTextPrompt": "extras.SubsequentTextPrompt",
    "Description": "extras.Description",
    "Score": "extras.Score"
}

fixed_values_answer = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "LOINC Answer",
    "datatype": "N/A",
    "source": global_source,
    "owner_type": "Organization",
    "owner": global_organization,
    "extras.Code_Type": "LOINC Answer"
}

In [ ]:
#Transformation rules - specify fields to transform and their target values

transformation_rules = [
    {
      "field": ["STATUS", "Status"],
      "transformations": {
        "DEPRECATED": True,
        "ACTIVE": False,
        "TRIAL": False,
        "DISCOURAGED": False
      },
      "target_field": "retired"
    },
    {
      "field": "COMMON_TEST_RANK",
      "transformations": {
        "0": None,  # Convert "0" to None so it gets filtered out
        "": None    # Also handle empty strings
      },
      "target_field": "extras.COMMON_TEST_RANK"
    },
    {
    "field": "COMMON_ORDER_RANK",
    "transformations": {
      # Convert decimal strings to integers
      lambda x: str(int(float(x))) if x and x != "0" and x != "0.0" else None
    },
    "target_field": "extras.COMMON_ORDER_RANK"
    }
]

In [ ]:
### Mapping Configurations ###

# Define the base URL for OCL concepts
CONCEPT_URL_PREFIX = f"/orgs/{global_organization}/sources/{global_source}/concepts/"

# Define the overall configuration dictionary
MAPPING_CONFIGS = {
    "Panel-to-Test": {
        "source_files": ["PanelsAndForms.csv"],
        "map_type": "Has Element",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Has Element"},
            {"ocl_field": "from_concept_url", "source_field": "ParentLoinc", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "to_concept_url", "source_field": "Loinc", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "source", "source_value": global_source},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": global_organization},
            {"ocl_field": 'extras["Sequence"]', "source_field": "SequenceInPanel"},
            {"ocl_field": 'extras["Required"]', "source_field": "Required"},
            {
                "ocl_field": 'extras["Cardinality"]',
                "source_field_min": "CardinalityMin",
                "source_field_max": "CardinalityMax",
                "rule": lambda min_val, max_val: f"{min_val}..{max_val}",
            },
            {"ocl_field": 'extras["Answer List Override"]', "source_field": "AnswerListIdOverride"},
            {"ocl_field": 'extras["Answer List Type Override"]', "source_field": "AnswerListTypeOverride"},
        ],
    },
    "Question-to-Answer": {
        "source_files": ["LoincAnswerListLink.csv", "AnswerList.csv"],
        "map_type": "Has Answer",
        "join_condition": {"left_on": "LoincAnswerListLink.AnswerListId", "right_on": "AnswerList.AnswerListId"},
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Has Answer"},
            {"ocl_field": "from_concept_url", "source_field": "LoincNumber", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "to_concept_url", "source_field": "AnswerStringId", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "source", "source_value": global_source},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": global_organization},
            {"ocl_field": 'extras["Answer List ID"]', "source_field": "LoincAnswerListLink.AnswerListId"},
            {"ocl_field": 'extras["Answer List Type"]', "source_field": "LoincAnswerListLink.AnswerListLink Type"},
            {"ocl_field": 'extras["Sequence"]', "source_field": "AnswerList.SequenceNumber"},
            {"ocl_field": 'extras["Score"]', "source_field": "AnswerList.Score"},
            {"ocl_field": 'extras["Local Answer Code"]', "source_field": "AnswerList.LocalAnswerCode"},
        ],
    },
    "Ask at Order Entry": {
        "source_files": ["Loinc.csv"],
        "map_type": "Ask At Order Entry",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Ask At Order Entry"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "to_concept_url", "source_field": "AskAtOrderEntry", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "source", "source_value": global_source},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": global_organization},
        ],
    },
    "Code Evolution": {
        "source_files": ["MapTo.csv"],
        "map_type": "Map To",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Map To"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "to_concept_url", "source_field": "MAP_TO", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "source", "source_value": global_source},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": global_organization},
            {"ocl_field": 'extras["COMMENT"]', "source_field": "COMMENT"},
        ],
    },
    "Associated Observations": {
        "source_files": ["Loinc.csv"],
        "map_type": "Associated Observations",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Associated Observations"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "to_concept_url", "source_field": "AssociatedObservations", "rule": lambda x: f"{CONCEPT_URL_PREFIX}{x}/"},
            {"ocl_field": "source", "source_value": global_source},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": global_organization},
        ],
    },
}

In [ ]:
###  Hierarchy Configurations  ###

hierarchy_mapping = {

    #Field Mappings
    # PATH_TO_ROOT,SEQUENCE,IMMEDIATE_PARENT,CODE,CODE_TEXT

    'PATH_TO_ROOT': 'extras.PATH_TO_ROOT',
    'SEQUENCE': 'extras.HIERARCHY_SEQUENCE',
    'IMMEDIATE_PARENT': 'parent_concept', # This will become the parent concept URL
    'CODE': 'match-id' # This will be used to match with the LOINC or Part concept ID
    # 'CODE_TEXT' will be ignored
}

## Phase 1: Data Loading and Validation

In [ ]:
# Data Loading and Validation - Concepts

def load_data():
    """Loads data based on the `mode` variable and validates fields against the mapping."""
    if mode == 2:
        loinc_df = demo_loinc_df
        part_link_df = demo_part_link_df
        answer_list_df = demo_answer_list_df
    elif mode == 1:
        # In test mode, we load a small subset of the real data
        try:
            loinc_df = pd.read_csv(loinc_csv_path, nrows=15, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")

        part_link_df = pd.read_csv(part_link_csv_path, nrows=15, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, nrows=15, low_memory=False)
    elif mode == 0:
        # In full run mode, we load the entire CSV files
        try:
            loinc_df = pd.read_csv(loinc_csv_path, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")

        part_link_df = pd.read_csv(part_link_csv_path, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, low_memory=False)
    else:
        print("Invalid mode selected. Please use 0, 1, or 2.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # ... (rest of the load_data function remains the same) ...
    # The validation logic below is not changed.

    # A helper function to create a DataFrame from the raw CSV data
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))

    # This part needs to be changed. The code below should dynamically
    # load files from the linguistic_variants_path directory for modes 0 and 1.
    if mode in [0, 1]:
        print("Loading linguistic variant files...")
        linguistic_variant_dfs = []
        for filename in os.listdir(linguistic_variants_path):
            if filename.endswith('.csv'):
                filepath = os.path.join(linguistic_variants_path, filename)
                df = pd.read_csv(filepath, low_memory=False)
                linguistic_variant_dfs.append(df)
    else: # mode == 2
        # Use hardcoded data for demo mode
        esMX_df = load_csv_data(esMX_data)
        etEE_df = load_csv_data(etEE_data)
        frBE_df = load_csv_data(frBE_data)
        linguistic_variant_dfs = [esMX_df, etEE_df, frBE_df]

    print(f"Loaded {len(linguistic_variant_dfs)} linguistic variant files.")

    return loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs

# Call the function to load all data
loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs = load_data()

def load_and_preprocess_hierarchy_data(file_path, column_map):
    """
    Loads the ComponentHierarchyBySystem CSV, renames and selects columns
    based on the provided mapping, and handles initial data types.
    """
    try:
        df = pd.read_csv(file_path)

        # Select only the columns specified in the mapping.
        # The key represents the new column name, and the value is the old column name.
        df = df[list(column_map.keys())]

        # Rename the selected columns using the mapping.
        df = df.rename(columns=column_map)

        return df
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

# Load the data into a DataFrame.
hierarchy_df = load_and_preprocess_hierarchy_data(component_hierarchy_file_path, hierarchy_mapping)

if hierarchy_df is not None:
    print("Successfully loaded and renamed hierarchy data with the new mapping:")
    # print(hierarchy_df.head())

In [ ]:
# Adds linguistic variant file names, dynamically generated from the directory.

# Function to check if a file is a proper linguistic variant file
def is_proper_linguistic_variant_file(filepath):
    """
    Checks if a file is a proper linguistic variant file by:
    1. Checking if filename matches the pattern xxYY##LinguisticVariant.csv
    2. Checking if it has the expected columns
    """
    filename = os.path.basename(filepath)

    # Skip the index file
    if filename == 'LinguisticVariants.csv':
        return False

    # Check if filename matches the expected pattern
    # Pattern: xxYY##LinguisticVariant.csv where xx=language, YY=country, ##=number
    pattern = r'^[a-z]{2}[A-Z]{2}\d+LinguisticVariant\.csv$'
    if not re.match(pattern, filename):
        return False

    try:
        # Check if file has the expected columns
        df_sample = pd.read_csv(filepath, nrows=1)
        expected_columns = ['LOINC_NUM', 'COMPONENT', 'PROPERTY', 'TIME_ASPCT',
                          'SYSTEM', 'SCALE_TYP', 'METHOD_TYP', 'CLASS']

        # Check if at least the core columns exist
        missing_columns = [col for col in expected_columns if col not in df_sample.columns]
        if missing_columns:
            print(f"⚠️  Skipping {filename}: missing columns {missing_columns}")
            return False

        return True
    except Exception as e:
        print(f"⚠️  Error checking {filename}: {e}")
        return False

# Function to create the Fully Specified Name (FSN) with error handling
def create_fsn(row):
    """
    Creates FSN from row data with proper error handling
    """
    try:
        # List of columns to check for FSN creation
        fsn_columns = ['COMPONENT', 'PROPERTY', 'TIME_ASPCT', 'SYSTEM', 'SCALE_TYP', 'METHOD_TYP', 'CLASS']

        # Check if all required columns exist
        missing_cols = [col for col in fsn_columns if col not in row.index]
        if missing_cols:
            # If missing critical columns, return None
            return None

        parts = [row[col] for col in fsn_columns]
        valid_parts = [str(part) for part in parts if pd.notna(part) and str(part).strip() != '']

        return ':'.join(valid_parts) if valid_parts else None

    except Exception as e:
        print(f"⚠️  Error creating FSN for row: {e}")
        return None

# Function to process a single linguistic variant file with error handling
def process_linguistic_file(file_path):
    """
    Process a single linguistic variant file and add the locale code
    """
    filename = os.path.basename(file_path)

    # Extract locale code from filename (first 4 characters: xxYY)
    if len(filename) >= 4:
        # Extract language (first 2 chars) and country (next 2 chars) separately
        language_code = filename[:2].lower()   # e.g., "es" from "esMX28LinguisticVariant.csv"
        country_code = filename[2:4].upper()   # e.g., "MX" from "esMX28LinguisticVariant.csv"
        locale_code = f"{language_code}-{country_code}"  # e.g., "es-MX"
    else:
        locale_code = filename[:2].lower()  # fallback to just language code

    print(f"Processing {filename} with locale code: {locale_code}")  # e.g., "fr-FR"

    try:
        df = pd.read_csv(file_path, low_memory=False)
        processed_rows = []

        # Process each row
        for index, row in df.iterrows():
            # Try to create FSN
            fsn = create_fsn(row)
            if fsn:
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': fsn,
                    'NameType': 'Fully-Specified'
                })

            # Process SHORTNAME if available
            if 'SHORTNAME' in row and pd.notna(row['SHORTNAME']) and str(row['SHORTNAME']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['SHORTNAME'],
                    'NameType': 'Short'
                })

            # Process LONG_COMMON_NAME if available
            if 'LONG_COMMON_NAME' in row and pd.notna(row['LONG_COMMON_NAME']) and str(row['LONG_COMMON_NAME']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['LONG_COMMON_NAME'],
                    'NameType': 'Display'
                })

            # Process RELATEDNAMES2 if available
            if 'RELATEDNAMES2' in row and pd.notna(row['RELATEDNAMES2']) and str(row['RELATEDNAMES2']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['RELATEDNAMES2'],
                    'NameType': 'None'
                })

        print(f"  ✅ Processed {len(processed_rows)} linguistic variant entries from {filename}")
        return pd.DataFrame(processed_rows)

    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")
        return pd.DataFrame()  # Return empty DataFrame on error

# Main linguistic variant processing
def process_all_linguistic_variants(linguistic_variants_path):
    """
    Process all valid linguistic variant files in the directory
    """
    print("🌍 Processing linguistic variant files...")

    # Get all CSV files in the directory
    all_csv_files = [
        os.path.join(linguistic_variants_path, f)
        for f in os.listdir(linguistic_variants_path)
        if f.endswith('.csv')
    ]

    # Filter to only proper linguistic variant files
    valid_linguistic_files = [
        filepath for filepath in all_csv_files
        if is_proper_linguistic_variant_file(filepath)
    ]

    print(f"Found {len(all_csv_files)} CSV files, {len(valid_linguistic_files)} valid linguistic variant files")

    if not valid_linguistic_files:
        print("⚠️  No valid linguistic variant files found!")
        return pd.DataFrame()

    # Process each valid file
    all_processed_variants = []
    for file_path in valid_linguistic_files:
        processed_df = process_linguistic_file(file_path)
        if not processed_df.empty:
            all_processed_variants.append(processed_df)

    # Concatenate all results
    if all_processed_variants:
        final_df = pd.concat(all_processed_variants, ignore_index=True)
        print(f"🎉 Successfully processed {len(final_df)} total linguistic variant entries")
        return final_df
    else:
        print("⚠️  No linguistic variant data was successfully processed")
        return pd.DataFrame()

# Replace the existing linguistic variant processing code with this:
if mode in [0, 1]:
    # Process real linguistic variant files
    all_processed_variants = process_all_linguistic_variants(linguistic_variants_path)
else:
    # Use demo data for mode 2
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))

    # Demo data processing (existing code can remain the same)
    esMX_df = load_csv_data(esMX_data)
    etEE_df = load_csv_data(etEE_data)
    frBE_df = load_csv_data(frBE_data)

    processed_demo_data = []
    for df, locale in [(esMX_df, 'es-MX'), (etEE_df, 'et-EE'), (frBE_df, 'fr-BE')]:
        for _, row in df.iterrows():
            fsn = create_fsn(row)
            if fsn:
                processed_demo_data.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale,
                    'LinguisticVariantName': fsn,
                    'NameType': 'Fully-Specified'
                })

    all_processed_variants = pd.DataFrame(processed_demo_data)

# Continue with the existing pivot logic
if not all_processed_variants.empty:
    # Create a copy to avoid modifying the original dataframe
    temp_df = all_processed_variants.copy()

    # Sort the data to ensure the numeric counter is assigned consistently
    temp_df.sort_values(by=['LOINC_NUM', 'NameType', 'LocaleCode'], inplace=True)

    # Generate a numeric counter for each unique combination
    temp_df['counter'] = temp_df.groupby(['LOINC_NUM', 'NameType', 'LocaleCode']).cumcount() + 1

    # Create pivot column
    temp_df['pivot_column'] = ('names.' + temp_df['NameType'].astype(str) + '.' +
                              temp_df['LocaleCode'].astype(str) + '[' +
                              temp_df['counter'].astype(str) + ']')

    # Pivot the DataFrame
    pivoted_variants = temp_df.pivot(index='LOINC_NUM',
                                   columns='pivot_column',
                                   values='LinguisticVariantName')

    # Flatten and reset index
    pivoted_variants.columns = pivoted_variants.columns.get_level_values(0)
    pivoted_variants.reset_index(inplace=True)

    # Merge with the main LOINC DataFrame
    merged_loinc_df = pd.merge(loinc_df, pivoted_variants, on='LOINC_NUM', how='left')

    print(f"✅ Merged linguistic variants with {len(merged_loinc_df)} LOINC concepts")
else:
    # No linguistic variants to merge
    merged_loinc_df = loinc_df.copy()
    print("ℹ️  No linguistic variants to merge - using original LOINC data only")

## Phase 2: Concept Creation

In this phase, we will transform LOINC terms, parts, answer lists, and linguistic variants into OCL Concept objects.

In [ ]:
## Functions for creating OCL concepts for all LOINC types

def check_for_unmapped_fields(df, mapping, name_of_df, ignored_cols=None):
    """
    Checks for unmapped columns in a DataFrame and prints a warning message.
    """
    if ignored_cols is None:
        ignored_cols = []

    mapped_source_fields = set(mapping.keys())

    # Get all columns from the DataFrame that are not in the ignored list
    df_columns = set(df.columns) - set(ignored_cols)

    unmapped_fields = df_columns - mapped_source_fields

    if unmapped_fields:
        print(f"⚠️ Warning: Unmapped fields found in '{name_of_df}': {', '.join(sorted(list(unmapped_fields)))}")

def create_ocl_concept_from_row(row, mapping, fixed_values, transformation_rules=None):
    """
    Creates a single OCL Concept dictionary from a pandas DataFrame row,
    applying the specified mapping, fixed values, and transformation rules.
    This version keeps all attributes as top-level fields.
    """
    concept = fixed_values.copy()

    # Process column mappings
    for source_field, target_fields in mapping.items():
        if source_field in row and pd.notna(row[source_field]):

            # Omit COMMON_TEST_RANK if the value is 0
            if source_field == 'COMMON_TEST_RANK' and row[source_field] == 0:
                continue

            # Omit COMMON_ORDER_RANK if the value is 0
            if source_field == 'COMMON_ORDER_RANK' and row[source_field] == 0:
                continue

            # Check if the value is a float and can be safely converted to an integer
            is_int_candidate = isinstance(row[source_field], float) and row[source_field].is_integer()

            # Special handling for specified fields that should be integers
            if source_field in ['CLASSTYPE', 'COMMON_ORDER_RANK', 'COMMON_TEST_RANK', 'HIERARCHY_SEQUENCE'] and is_int_candidate:
                value = int(row[source_field])
            else:
                value = row[source_field]

            if isinstance(target_fields, list):
                for target_field in target_fields:
                    concept[target_field] = value
            else:
                concept[target_fields] = value

    # Process transformation rules
    if transformation_rules:
        for rule in transformation_rules:
            fields_to_check = rule['field'] if isinstance(rule['field'], list) else [rule['field']]

            for field in fields_to_check:
                # Check for the existence of the field and handle both existing and missing values
                if field in row:
                    source_value = row[field]

                    # Special handling for NaN, which cannot be a dictionary key
                    # Check if the value is NaN and if the rule has a transformation for NaN
                    if pd.isna(source_value):
                        if np.nan in rule['transformations']:
                            target_value = rule['transformations'][np.nan]
                            concept[rule['target_field']] = target_value
                            break # Stop after finding the first matching field and transforming
                    elif source_value in rule['transformations']:
                        target_value = rule['transformations'][source_value]
                        concept[rule['target_field']] = target_value
                        break # Stop after finding the first matching field and transforming

    return concept

def process_loinc_concepts(df):
    """
    Creates OCL Concept objects for all LOINC codes, including linguistic variants,
    with all attributes as top-level fields.
    """
    loinc_concepts = []

    linguistic_variant_cols = [col for col in df.columns if col.startswith('names.')]

    # Check for unmapped fields in the DataFrame
    ignored_cols = linguistic_variant_cols + ['PartNumber', 'ANSWERLISTID']
    check_for_unmapped_fields(df, loinc_to_ocl_mapping, 'merged_loinc_df', ignored_cols=ignored_cols)

    for _, row in df.iterrows():
        base_loinc_concept = create_ocl_concept_from_row(row, loinc_to_ocl_mapping, fixed_values_loinc, transformation_rules)

        # Handle dynamic linguistic variant names
        for col in linguistic_variant_cols:
            if pd.notna(row[col]):
                base_loinc_concept[col] = row[col]

        loinc_concepts.append(base_loinc_concept)

    return loinc_concepts

def process_loinc_parts(df):
    """
    Creates OCL Concept objects for unique LOINC Parts with all attributes
    as top-level fields.
    """
    loinc_part_concepts = []
    processed_part_numbers = set()

    # Check for unmapped fields
    check_for_unmapped_fields(df, loinc_part_to_ocl_mapping, 'part_link_df')

    for _, row in df.iterrows():
        part_number = row['PartNumber']
        if part_number not in processed_part_numbers:
            concept = create_ocl_concept_from_row(
                row, loinc_part_to_ocl_mapping, fixed_values_loinc_parts, transformation_rules
            )
            loinc_part_concepts.append(concept)
            processed_part_numbers.add(part_number)

    return loinc_part_concepts

def process_answer_lists_and_answers(df):
    """
    Creates OCL Concept objects for both LOINC Answer Lists and individual Answers,
    with all attributes as top-level fields.
    Deduplicates concepts to ensure uniqueness.
    """
    answer_list_concepts = []
    answer_concepts = []
    processed_answer_list_ids = set()
    processed_answer_ids = set()

    # Check for unmapped fields
    check_for_unmapped_fields(df, {**answer_list_to_ocl_mapping, **answer_to_ocl_mapping}, 'answer_list_df')

    for _, row in df.iterrows():
        # Create Answer List Concept
        answer_list_id = row['AnswerListId']
        if answer_list_id not in processed_answer_list_ids:
            answer_list_concept = create_ocl_concept_from_row(
                row, answer_list_to_ocl_mapping, fixed_values_answer_list, transformation_rules
            )
            answer_list_concepts.append(answer_list_concept)
            processed_answer_list_ids.add(answer_list_id)

        # Create Answer Concept
        answer_string_id = row['AnswerStringId']
        if answer_string_id not in processed_answer_ids:
            answer_concept = create_ocl_concept_from_row(
                row, answer_to_ocl_mapping, fixed_values_answer, transformation_rules
            )
            answer_concept['ParentAnswerListId'] = answer_list_id

            answer_concepts.append(answer_concept)
            processed_answer_ids.add(answer_string_id)

    return answer_list_concepts, answer_concepts

In [ ]:
def phase_2_main(merged_loinc_df, part_link_df, answer_list_df):
    """
    Main function to execute all Phase 2 concept creation tasks and
    consolidate the outputs into a single list.
    """
    # Create LOINC Concepts
    print("Creating LOINC concepts...")
    loinc_concepts = process_loinc_concepts(merged_loinc_df)
    print(f"Created {len(loinc_concepts)} LOINC concepts from `merged_loinc_df`.")

    # Create LOINC Part Concepts
    print("Creating LOINC Part concepts...")
    loinc_part_concepts = process_loinc_parts(part_link_df)
    print(f"Created {len(loinc_part_concepts)} LOINC Part concepts from `part_link_df`.")

    # Create Answer List and Answer Concepts
    print("Creating Answer List and Answer concepts...")
    answer_list_concepts, answer_concepts = process_answer_lists_and_answers(answer_list_df)
    print(f"Created {len(answer_list_concepts)} Answer List concepts and {len(answer_concepts)} Answer concepts from `answer_list_df`.")

    # Consolidate all concepts into a single list
    all_concepts = loinc_concepts + loinc_part_concepts + answer_list_concepts + answer_concepts

    return all_concepts

# Execute the main function of Phase 2
all_ocl_concepts_list = phase_2_main(merged_loinc_df, part_link_df, answer_list_df)

# Convert the list of concepts into a pandas DataFrame.
all_ocl_concepts_df = pd.DataFrame(all_ocl_concepts_list)

# Now, sort the DataFrame's columns alphabetically.
all_ocl_concepts_df_sorted = all_ocl_concepts_df.sort_index(axis=1)

# Print the total number of concepts created.
print(f"\nTotal OCL Concepts created in Phase 2: {len(all_ocl_concepts_df_sorted)}")

# The sorted DataFrame is now stored in `all_ocl_concepts_df_sorted`.

## Phase 3: Mapping Creation

This phase is for creating OCL Mapping objects based on relational files like `PanelsAndForms.csv` and `MapTo.csv`.

In [ ]:
# --- DATA LOADING AND PREPARATION ---


try:
    df_panels_and_forms = pd.read_csv(panels_and_forms_csv_path, low_memory=False)
    df_loinc_answer_list_link = pd.read_csv(loinc_answer_list_link_csv_path, low_memory=False)
    df_answer_list = pd.read_csv(answer_list_csv_path, low_memory=False)
    df_loinc = pd.read_csv(loinc_csv_path, low_memory=False)
    df_map_to = pd.read_csv(map_to_csv_path, low_memory=False)
    print("All DataFrames loaded successfully.")

except NameError:
    # Fallback to hardcoded file names if path variables are not defined.
    print("Warning: Path variables not found. Attempting to load from hardcoded filenames.")
    try:
        df_panels_and_forms = pd.read_csv("PanelsAndForms.csv")
        df_loinc_answer_list_link = pd.read_csv("LoincAnswerListLink.csv")
        df_answer_list = pd.read_csv("AnswerList.csv")
        df_loinc = pd.read_csv("Loinc.csv")
        df_map_to = pd.read_csv("MapTo.csv")
        print("DataFrames loaded from hardcoded filenames.")
    except FileNotFoundError as e:
        print(f"Error: One or more files not found: {e}")
        # Initialize empty DataFrames to prevent further errors
        df_panels_and_forms = pd.DataFrame()
        df_loinc_answer_list_link = pd.DataFrame()
        df_answer_list = pd.DataFrame()
        df_loinc = pd.DataFrame()
        df_map_to = pd.DataFrame()

except FileNotFoundError as e:
    print(f"Error: One or more files not found: {e}")
    # Initialize empty DataFrames to prevent further errors
    df_panels_and_forms = pd.DataFrame()
    df_loinc_answer_list_link = pd.DataFrame()
    df_answer_list = pd.DataFrame()
    df_loinc = pd.DataFrame()
    df_map_to = pd.DataFrame()

In [ ]:
## Mappings Processing

# Create a dictionary of global variables to pass to functions
global_variables = {
    "global_organization": global_organization,
    "global_source": global_source
}

# Process the mappings.
def process_mappings(df, config, globals_dict, debug=False):
    """
    Generates OCL mapping objects from a DataFrame using a configuration dictionary.
    """
    mapping_objects = []

    for index, row in df.iterrows():
        ocl_object = {}
        for field_map in config["field_mappings"]:
            try:
                ocl_field = field_map["ocl_field"]
                value = None
                rule = field_map.get("rule")

                if "source_value" in field_map:
                    value = field_map["source_value"]
                elif "source_field" in field_map:
                    source_field = field_map["source_field"]
                    source_value = row.get(source_field)

                    if pd.notna(source_value):
                        if rule:
                            if callable(rule):
                                value = rule(source_value)
                            else:
                                format_data = {**globals_dict, source_field: source_value}
                                value = rule.format(**format_data)
                        else:
                            value = source_value
                    # If value is NaN, 'value' remains None, which is the correct behavior
                elif "source_field_min" in field_map and "source_field_max" in field_map:
                    min_val = row.get(field_map["source_field_min"])
                    max_val = row.get(field_map["source_field_max"])
                    if pd.notna(min_val) and pd.notna(max_val) and rule:
                        if callable(rule):
                            value = rule(min_val, max_val)
                        else:
                            format_data = {**globals_dict, "min_val": min_val, "max_val": max_val}
                            value = rule.format(**format_data)
                    # If min or max are NaN, 'value' remains None
                else:
                    if debug:
                        print(f"Skipping malformed field_map at row {index}: {field_map}", file=sys.stderr)
                    continue

                if ocl_field.startswith('extras'):
                    if "extras" not in ocl_object:
                        ocl_object["extras"] = {}
                    try:
                        key = ocl_field.split('"')[1]
                        ocl_object["extras"][key] = value
                    except IndexError:
                        if debug:
                            print(f"Warning: Malformed 'extras' key '{ocl_field}' at row {index}.", file=sys.stderr)
                else:
                    ocl_object[ocl_field] = value

            except KeyError as e:
                if debug:
                    print(f"KeyError: {e} encountered at row {index} for field_map:", file=sys.stderr)
                    print(f"  Field Map: {field_map}", file=sys.stderr)
                    print(f"  Row Data: {row.to_dict()}", file=sys.stderr)
                continue
            except Exception as e:
                if debug:
                    print(f"An unexpected error occurred at row {index}: {e}", file=sys.stderr)
                    print(f"  Field Map: {field_map}", file=sys.stderr)
                continue

        if ocl_object.get("from_concept_url") and ocl_object.get("to_concept_url"):
            mapping_objects.append(ocl_object)

    return mapping_objects

# --- MAPPING EXECUTION ---

# Assuming your DataFrames are already loaded (e.g., df_loinc, df_panels_and_forms, etc.)

# Initialize mapping lists to empty
panel_to_test_mappings = []
qa_mappings = []
order_entry_mappings = []
code_evolution_mappings = []
associated_observations_mappings = []

# Process the Panel-to-Test mappings.
panel_to_test_mappings = process_mappings(df_panels_and_forms, MAPPING_CONFIGS["Panel-to-Test"], global_variables)
print(f"Generated {len(panel_to_test_mappings)} Panel-to-Test mappings.")
print("-" * 20)

# Process the Question-to-Answer mappings.
join_info = MAPPING_CONFIGS["Question-to-Answer"]["join_condition"]
left_col = join_info["left_on"].split('.')[-1]
right_col = join_info["right_on"].split('.')[-1]
df_joined_qa = pd.merge(df_loinc_answer_list_link, df_answer_list, left_on=left_col, right_on=right_col, suffixes=("_LoincAnswerListLink", "_AnswerList"))
qa_mappings = process_mappings(df_joined_qa, MAPPING_CONFIGS["Question-to-Answer"], global_variables)
print(f"Generated {len(qa_mappings)} Question-to-Answer mappings.")
print("-" * 20)

# Process the Ask at Order Entry mappings.
order_entry_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Ask at Order Entry"], global_variables)
print(f"Generated {len(order_entry_mappings)} Ask at Order Entry mappings.")
print("-" * 20)

# Process the Code Evolution mappings.
code_evolution_mappings = process_mappings(df_map_to, MAPPING_CONFIGS["Code Evolution"], global_variables)
print(f"Generated {len(code_evolution_mappings)} Code Evolution mappings.")
print("-" * 20)

# Process the Associated Observations mappings.
def preprocess_associated_observations(df):
    """
    Preprocesses the LOINC DataFrame to expand rows with delimited AssociatedObservations values.
    Each semicolon-separated LOINC code becomes its own row for mapping creation.

    Args:
        df: DataFrame containing LOINC data with AssociatedObservations column

    Returns:
        Expanded DataFrame with individual rows for each associated observation
    """
    import re

    def is_valid_loinc_format(code):
        """Check if a string matches basic LOINC format (NNNNN-N)"""
        if not code or pd.isna(code):
            return False
        code = str(code).strip()
        # LOINC pattern: 1-5 digits, hyphen, 1-2 digits
        return bool(re.match(r'^\d{1,5}-\d{1,2}$', code))

    def extract_loinc_codes(associated_obs_value):
        """Extract valid LOINC codes from AssociatedObservations field"""
        if pd.isna(associated_obs_value):
            return []

        # Convert to string and split on common delimiters
        text = str(associated_obs_value).strip()
        if not text:
            return []

        # Split on semicolons, commas, or pipe characters
        potential_codes = re.split(r'[;,|]', text)

        valid_codes = []
        for code in potential_codes:
            code = code.strip()
            if is_valid_loinc_format(code):
                valid_codes.append(code)

        return valid_codes

    # Filter to only rows with AssociatedObservations values
    has_assoc_obs = df['AssociatedObservations'].notna() & (df['AssociatedObservations'].astype(str).str.strip() != '')
    df_with_assoc = df[has_assoc_obs].copy()

    if df_with_assoc.empty:
        print("No rows found with AssociatedObservations values")
        return pd.DataFrame()

    print(f"Found {len(df_with_assoc)} rows with AssociatedObservations values")

    # Expand rows with multiple associated observations
    expanded_rows = []
    delimited_count = 0
    total_mappings = 0

    for _, row in df_with_assoc.iterrows():
        associated_codes = extract_loinc_codes(row['AssociatedObservations'])

        if len(associated_codes) > 1:
            delimited_count += 1

        for code in associated_codes:
            # Create a new row for each associated observation
            new_row = row.copy()
            new_row['AssociatedObservations'] = code
            expanded_rows.append(new_row)
            total_mappings += 1

    if not expanded_rows:
        print("No valid LOINC codes found in AssociatedObservations fields")
        return pd.DataFrame()

    expanded_df = pd.DataFrame(expanded_rows)

    print(f"Expanded {delimited_count} rows with delimited values")
    print(f"Created {total_mappings} individual mapping rows")
    # print(f"Sample expanded AssociatedObservations values: {list(expanded_df['AssociatedObservations'].head(10))}")

    return expanded_df


print("=== Processing Associated Observations Mappings ===")

# Expand the LOINC dataframe for Associated Observations
df_loinc_expanded_assoc = preprocess_associated_observations(df_loinc)

if not df_loinc_expanded_assoc.empty:
    # Process the expanded dataframe through the normal mapping process
    associated_observations_mappings = process_mappings(df_loinc_expanded_assoc, MAPPING_CONFIGS["Associated Observations"], global_variables)
    print(f"Generated {len(associated_observations_mappings)} Associated Observations mappings.")
else:
    # Fallback to original processing if no expansion occurred
    print("No delimited values found, using original processing...")
    associated_observations_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Associated Observations"])
    print(f"Generated {len(associated_observations_mappings)} Associated Observations mappings.")

print("-" * 20)

# Combine all mappings into a single list for the final output
all_mappings = (
    panel_to_test_mappings +
    qa_mappings +
    order_entry_mappings +
    code_evolution_mappings +
    associated_observations_mappings
)
print(f"\nTotal OCL mapping objects created for Round 1: {len(all_mappings)}")
mappings_df = pd.DataFrame(all_mappings)

In [ ]:
# --- Advanced Deduplication: Group, Aggregate, and Merge ---

print("Applying advanced deduplication: grouping and merging...")
initial_mapping_count = len(mappings_df)
print(f"Number of mappings before aggregation: {initial_mapping_count}")

# Define the columns that identify a unique mapping group
key_mapping_columns = [
    'from_concept_url',
    'to_concept_url',  'map_type'
    # NOTE: Ensure all columns that define a unique group are listed here.
    # Exclude columns you want to aggregate, like 'extras'.
]

# Define the custom function to aggregate the 'extras' column
def merge_extras(group):
    # If there's only one row in the group, no need to merge
    if len(group) == 1:
        return group.iloc[0]

    # Use the first row as the base for our merged record
    merged_row = group.iloc[0].copy()

    all_overrides = []

    # Iterate over each row in the duplicate group
    for index, row in group.iterrows():
        extras_val = row.get('extras')
        if pd.isna(extras_val) or not isinstance(extras_val, str) or extras_val.strip() == '':
            continue

        try:
            # Safely parse the string into a dictionary
            extras_dict = ast.literal_eval(extras_val)
            override = extras_dict.get('Answer List Override')

            # Add the override value if it's not null/empty and not already in our list
            if override and str(override).strip() and override not in all_overrides:
                all_overrides.append(str(override))
        except (ValueError, SyntaxError):
            # Handle cases where the string is not a valid dict
            continue

    # If we found any overrides, merge them and update the 'extras' field
    if all_overrides:
        try:
            # Reconstruct the 'extras' dictionary
            final_extras_dict = ast.literal_eval(merged_row['extras'])
            final_extras_dict['Answer List Override'] = ", ".join(all_overrides)
            merged_row['extras'] = str(final_extras_dict)
        except (ValueError, SyntaxError):
            # Fallback if the base row's 'extras' is invalid
            pass

    return merged_row

# Group by the key columns and apply our merging function
# This is the core of the aggregation logic
merged_mappings_df = mappings_df.groupby(key_mapping_columns, as_index=False).apply(merge_extras)

# Reset the index to be clean
merged_mappings_df.reset_index(drop=True, inplace=True)

# Final count and report
final_mapping_count = len(merged_mappings_df)
print(f"Number of mappings after aggregation: {final_mapping_count}")
print(f"Consolidated {initial_mapping_count - final_mapping_count} duplicate rows.")

# IMPORTANT: Overwrite the old DataFrame with the new, merged one
mappings_df = merged_mappings_df
# --- End of Aggregation Logic ---

In [ ]:
# --- Data Quality Check: Find Conflicting Map Types for Any Mapping ---

print("\\n🔎 Checking for conflicting map types (same from/to, different map_type)...")

# 1. Group by the mapping pair (from_concept_url and to_concept_url)
#    Then, aggregate to find the number of unique map_types and list them.
conflict_check = mappings_df.groupby(['from_concept_url', 'to_concept_url'])['map_type'].agg(['nunique', 'unique']).reset_index()

# 2. Filter to find the actual conflicts (where there's more than 1 unique map type)
conflicts = conflict_check[conflict_check['nunique'] > 1]

# 3. Alert the user if any conflicts were found
if not conflicts.empty:
    print("🚨 WARNING: Found mappings with the same 'from' and 'to' concepts but conflicting map_types!")
    print("   This can indicate a data inconsistency. Please review the following pairs:")
    print("-" * 80)

    for index, row in conflicts.iterrows():
        from_url = row['from_concept_url']
        to_url = row['to_concept_url']
        map_types = row['unique']

        print(f"  - From: {from_url}")
        print(f"    To:   {to_url}")
        print(f"    Conflicting Map Types: {', '.join(map_types)}")
        print() # Add a blank line for readability

    print("-" * 80)
else:
    print("✅ No map type conflicts found.")

# --- End of Data Quality Check ---

## Phase 4: Hierarchy Creation

In [ ]:
### Hierarchy Analysis Functions ###

def identify_node_types(df):
    """
    Identifies LOINC Parts (branches) and LOINC terms (leaf nodes) in the hierarchy.
    LOINC Parts are the branches, and LOINC terms are the leaf nodes.
    """
    # LOINC Parts (branches) appear in the 'parent_concept' column.
    loinc_parts = df['parent_concept'].dropna().unique()
    # LOINC terms (leaf nodes) are 'match-id' values that are not 'parent_concept' values.
    loinc_terms = df[~df['match-id'].isin(loinc_parts)]['match-id'].unique()

    print("\nLOINC Hierarchy Breakdown:")
    print(f"Total unique codes: {df['match-id'].nunique()}")
    print(f"Number of LOINC Parts (branches): {len(loinc_parts)}")
    print(f"Number of LOINC Terms (leaf nodes): {len(loinc_terms)}")
    return loinc_parts, loinc_terms

# Execute the analysis functions if data was loaded successfully.
if hierarchy_df is not None:
    loinc_parts, loinc_terms = identify_node_types(hierarchy_df)
else:
    print("Error: The hierarchy_df is not available. Please check the data loading step.")

In [ ]:
# Merge and parent URL creation

print("🔧 Implementing Multi-Parent Hierarchy Support")
print("=" * 60)

# Step 1: Don't deduplicate hierarchy_df - we want to preserve all relationships
print(f"📊 Preserving all hierarchy relationships:")
print(f"   Total hierarchy records: {len(hierarchy_df):,}")
print(f"   Unique concepts: {hierarchy_df['match-id'].nunique():,}")
print(f"   Concepts with multiple parents: {len(hierarchy_df) - hierarchy_df['match-id'].nunique():,}")

# Step 2: Merge all concepts with all hierarchy records (this will create duplicates)
print(f"\n🔗 Merging concepts with hierarchy data...")
concepts_with_hierarchy_expanded = pd.merge(
    all_ocl_concepts_df_sorted,
    hierarchy_df,
    left_on='id',
    right_on='match-id',
    how='left'
)
print(f"   After merge: {len(concepts_with_hierarchy_expanded):,} rows (includes duplicates)")

# Step 3: Group by concept ID and aggregate parent concepts
print(f"\n🔄 Consolidating multiple parents per concept...")

def combine_parent_concepts(group):
    """
    Combine multiple parent concepts for a single concept,
    using ParentAnswerListId as a fallback.
    """
    base_row = group.iloc[0].copy()
    
    parent_concepts = []
    hierarchy_paths = []
    hierarchy_sequences = []

    for _, row in group.iterrows():
        # First, try to get the parent from the 'parent_concept' field
        parent_url = row.get('parent_concept')
        
        # If 'parent_concept' is missing, check for 'ParentAnswerListId'
        if pd.isna(parent_url) and pd.notna(row.get('ParentAnswerListId')):
            parent_list_id = row['ParentAnswerListId']
            # Assume CONCEPT_URL_PREFIX is defined elsewhere, like it is in the notebook
            parent_url = f"{CONCEPT_URL_PREFIX}{parent_list_id}/"

        if pd.notna(parent_url):
            parent_concepts.append(str(parent_url))
        
        # Original logic for hierarchy paths and sequences remains the same
        if pd.notna(row.get('extras.PATH_TO_ROOT')):
            hierarchy_paths.append(str(row['extras.PATH_TO_ROOT']))
        if pd.notna(row.get('extras.HIERARCHY_SEQUENCE')):
            hierarchy_sequences.append(row['extras.HIERARCHY_SEQUENCE'])

    unique_parents = list(dict.fromkeys(parent_concepts))
    unique_paths = list(dict.fromkeys(hierarchy_paths))
    unique_sequences = list(dict.fromkeys(hierarchy_sequences))
    
    if unique_parents:
        if len(unique_parents) > 1:
            base_row['parent_concept'] = unique_parents
        else:
            base_row['parent_concept'] = unique_parents[0]
    else:
        base_row['parent_concept'] = None
    
    if unique_paths:
        if len(unique_paths) > 1:
            base_row['extras.PATH_TO_ROOT'] = unique_paths
        else:
            base_row['extras.PATH_TO_ROOT'] = unique_paths[0]
    else:
        base_row['extras.PATH_TO_ROOT'] = None
    
    if unique_sequences:
        if len(unique_sequences) > 1:
            base_row['extras.HIERARCHY_SEQUENCE'] = unique_sequences
        else:
            base_row['extras.HIERARCHY_SEQUENCE'] = unique_sequences[0]
    else:
        base_row['extras.HIERARCHY_SEQUENCE'] = None
    
    return base_row


# Apply the grouping and consolidation
concepts_with_hierarchy_df = concepts_with_hierarchy_expanded.groupby('id').apply(combine_parent_concepts).reset_index(drop=True)

print(f"   After consolidation: {len(concepts_with_hierarchy_df):,} rows (one per concept)")

# Step 4: Create parent_concept_urls supporting multiple parents
def create_parent_urls_multi(row):
    """
    Creates a list of parent concept URLs for a given row, supporting multiple parents.
    FIXED: Properly handles NaN values to prevent "nan" string parents.
    """
    parent_urls = []

    # Handle hierarchy parent(s)
    parent_concept = row.get('parent_concept')

    if parent_concept is not None:
        if isinstance(parent_concept, list):
            # Multiple hierarchy parents
            for parent in parent_concept:
                if (parent is not None and
                    not pd.isna(parent) and
                    str(parent).strip() and
                    str(parent).lower() != 'nan'):  # Exclude "nan" strings
                    parent_urls.append(f"{CONCEPT_URL_PREFIX}{parent_concept}/")
        else:
            # Single hierarchy parent
            if (parent_concept is not None and
                not pd.isna(parent_concept) and
                str(parent_concept).strip() and
                str(parent_concept).lower() != 'nan'):  # Exclude "nan" strings
                parent_urls.append(f"{CONCEPT_URL_PREFIX}{parent_concept}/")

    # Handle Answer List parent (from Phase 2) - FIXED NaN handling
    answer_list_parent = row.get('ParentAnswerListId')
    if (answer_list_parent is not None and
        not pd.isna(answer_list_parent) and
        str(answer_list_parent).strip() and
        str(answer_list_parent).lower() != 'nan'):  # Exclude "nan" strings
        parent_urls.append(f"{CONCEPT_URL_PREFIX}{parent_concept}/")

    # Return the list of parent URLs, or None if empty
    return parent_urls if parent_urls else None

# Apply the multi-parent URL creation
concepts_with_hierarchy_df['parent_concept_urls'] = concepts_with_hierarchy_df.apply(create_parent_urls_multi, axis=1)

# Step 5: Report the results
print(f"\n📊 Multi-Parent Hierarchy Results:")
print(f"   Total concepts: {len(concepts_with_hierarchy_df):,}")

# Count concepts by number of parents
parent_counts = concepts_with_hierarchy_df['parent_concept_urls'].apply(
    lambda x: len(x) if isinstance(x, list) else (1 if x is not None else 0)
).value_counts().sort_index()

print(f"   Concepts by number of parents:")
for num_parents, count in parent_counts.items():
    if num_parents == 0:
        print(f"     No parents: {count:,} concepts")
    elif num_parents == 1:
        print(f"     Single parent: {count:,} concepts")
    else:
        print(f"     {num_parents} parents: {count:,} concepts")

# Show examples of concepts with multiple parents
multi_parent_concepts = concepts_with_hierarchy_df[
    concepts_with_hierarchy_df['parent_concept_urls'].apply(
        lambda x: isinstance(x, list) and len(x) > 1
    )
]

if not multi_parent_concepts.empty:
    print(f"\n🔍 Examples of concepts with multiple parents:")
    for i, (_, row) in enumerate(multi_parent_concepts.head(5).iterrows()):
        concept_id = row['id']
        parents = row['parent_concept_urls']
        parent_ids = [url.split('/')[-2] for url in parents]  # Extract parent IDs from URLs
        print(f"     {concept_id}: {len(parents)} parents -> {', '.join(parent_ids)}")

print(f"\n✅ Multi-parent hierarchy implementation complete!")
print("=" * 60)

# Deduplicate to ensure no duplicate concepts (should be unnecessary but safety check)
concepts_with_hierarchy_df = concepts_with_hierarchy_df.drop_duplicates(subset=['id'], keep='first')

print(f"Final concept count: {len(concepts_with_hierarchy_df):,}")

## Phase 5: UMLS Enhancement

Here, we'll query an external UMLS API to enrich the concepts with CUIs (Concept Unique Identifiers).

In [ ]:
# # Analysis and Debug of LOINC Hierarchy - Optional

# print("🔍 DEBUGGING: Checking for duplicates in concepts_with_hierarchy_df")
# print("=" * 60)

# # Check the input dataframe for duplicates
# print(f"📊 concepts_with_hierarchy_df shape: {concepts_with_hierarchy_df.shape}")
# print(f"   Total rows: {len(concepts_with_hierarchy_df):,}")
# print(f"   Unique IDs: {concepts_with_hierarchy_df['id'].nunique():,}")

# # Check for duplicate IDs
# duplicate_ids = concepts_with_hierarchy_df[concepts_with_hierarchy_df['id'].duplicated(keep=False)]
# if not duplicate_ids.empty:
#     print(f"❌ FOUND {len(duplicate_ids):,} rows with duplicate IDs!")
#     print(f"   Number of unique duplicate IDs: {duplicate_ids['id'].nunique()}")

#     # Show examples of duplicates
#     print("\n🔍 Examples of duplicate IDs:")
#     sample_duplicates = duplicate_ids.groupby('id').size().head(10)
#     for concept_id, count in sample_duplicates.items():
#         print(f"   {concept_id}: {count} copies")

#     # Show details of first duplicate
#     first_duplicate_id = sample_duplicates.index[0]
#     duplicate_rows = concepts_with_hierarchy_df[concepts_with_hierarchy_df['id'] == first_duplicate_id]
#     print(f"\n📋 Details of duplicate ID '{first_duplicate_id}':")
#     print(f"   Columns that differ between duplicates:")

#     # Check which columns differ between the duplicate rows
#     for col in duplicate_rows.columns:
#         unique_values = duplicate_rows[col].nunique()
#         if unique_values > 1:
#             print(f"     {col}: {unique_values} different values")
#             print(f"       Values: {list(duplicate_rows[col].unique())}")

# else:
#     print("✅ No duplicate IDs found in concepts_with_hierarchy_df")

# # Check if the issue might be in hierarchy merge
# print("\n🔍 Checking hierarchy_df for issues...")
# if 'hierarchy_df' in globals():
#     print(f"📊 hierarchy_df shape: {hierarchy_df.shape}")
#     if 'match-id' in hierarchy_df.columns:
#         print(f"   Unique match-ids: {hierarchy_df['match-id'].nunique():,}")
#         duplicate_match_ids = hierarchy_df[hierarchy_df['match-id'].duplicated(keep=False)]
#         if not duplicate_match_ids.empty:
#             print(f"❌ hierarchy_df has {len(duplicate_match_ids):,} rows with duplicate match-ids!")
#             sample_hierarchy_dups = duplicate_match_ids['match-id'].value_counts().head(5)
#             print("   Examples:")
#             for match_id, count in sample_hierarchy_dups.items():
#                 print(f"     {match_id}: {count} copies")
#         else:
#             print("✅ No duplicate match-ids in hierarchy_df")

# # Check all_ocl_concepts_df_sorted (the original concepts)
# print("\n🔍 Checking all_ocl_concepts_df_sorted for issues...")
# if 'all_ocl_concepts_df_sorted' in globals():
#     print(f"📊 all_ocl_concepts_df_sorted shape: {all_ocl_concepts_df_sorted.shape}")
#     print(f"   Unique IDs: {all_ocl_concepts_df_sorted['id'].nunique():,}")

#     original_duplicates = all_ocl_concepts_df_sorted[all_ocl_concepts_df_sorted['id'].duplicated(keep=False)]
#     if not original_duplicates.empty:
#         print(f"❌ ORIGINAL concepts dataframe has {len(original_duplicates):,} duplicate IDs!")
#         print("   This is the root cause of the problem!")
#     else:
#         print("✅ Original concepts dataframe has no duplicates")

# print("=" * 60)
# print("🎯 DIAGNOSIS COMPLETE - Check the output above to identify the source")

In [ ]:
# UMLS Cache Setup

import json
import logging
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Optional
from datetime import datetime
import time
import requests

# Set up logging early
def setup_logging(log_level: str = "INFO"):
    """Setup logging based on config."""
    numeric_level = getattr(logging, log_level.upper(), logging.INFO)
    logging.basicConfig(
        level=numeric_level,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.getLogger(__name__).info("✅ Logging setup complete.")

# Utility Functions
# ============================================================================

def load_config(config_path: str = "UMLS_API_config.json") -> Dict[str, Any]:
    """Load configuration from JSON file."""
    # Modify the default output directory here to point to the content folder
    default_config = {
        "api_key": None,
        "rate_limit_delay": 0.1,
        "default_output_format": "json",
        "log_level": "INFO",
        "output_directory": "./", # Changed default to current directory (content in Colab)
        "settings": {
            "include_obsolete": False,
            "include_suppressible": False,
            "preferred_language": "ENG",
            "max_retries": 3
        }
    }

    if not Path(config_path).exists():
        logging.getLogger(__name__).warning(f"Config file not found: {config_path}. Using default settings.")
        return default_config
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
        logging.getLogger(__name__).info(f"Loaded configuration from {config_path}")
        # Ensure the output_directory is set correctly from config or falls back to default
        if "output_directory" not in config or not config["output_directory"]:
             config["output_directory"] = default_config["output_directory"]
        return config
    except Exception as e:
        logging.getLogger(__name__).error(f"Error loading config file {config_path}: {e}")
        # Return default config on error
        return default_config

# ... (rest of the code for setup_logging and load_config) ...

# UMLS LOINC Extractor Class

class UMLSLoincExtractor:
    """Extracts LOINC-to-CUI mappings from UMLS MRCONSO.RRF file."""

    def __init__(self, config: Optional[Dict[str, Any]] = None):
        """Initialize the extractor with configuration."""
        self.config = config or {}
        settings = self.config.get("settings", {})
        self.include_obsolete = settings.get("include_obsolete", False)
        self.include_suppressible = settings.get("include_suppressible", False)
        self.preferred_language = settings.get("preferred_language", "ENG")
        # Use the output_directory from the loaded/default config
        self.output_directory = Path(self.config.get("output_directory", "./")) # Ensure default points to current dir
        self.output_directory.mkdir(exist_ok=True)
        self.logger = logging.getLogger(__name__)

    def extract_from_file(self, mrconso_path: Path) -> Dict[str, str]:
        """
        Extract LOINC-to-CUI mappings from MRCONSO.RRF file.
        """
        if not mrconso_path.exists():
            raise FileNotFoundError(f"MRCONSO.RRF file not found: {mrconso_path}")

        self.logger.info(f"Starting LOINC extraction from: {mrconso_path}")
        start_time = datetime.now()

        try:
            column_names = [
                'CUI', 'LAT', 'TS', 'LUI', 'STT', 'SUI', 'ISPREF', 'AUI',
                'SAUI', 'SCUI', 'SDUI', 'SAB', 'TTY', 'CODE', 'STR', 'SRL',
                'SUPPRESS', 'CVF', 'EXTRA'
            ]

            chunk_size = 100000
            loinc_mappings = {}
            total_rows = 0
            loinc_rows = 0

            self.logger.info("Reading MRCONSO.RRF file in chunks...")

            for chunk_num, chunk in enumerate(pd.read_csv(
                mrconso_path,
                sep='|',
                names=column_names,
                dtype=str,
                na_filter=False,
                chunksize=chunk_size,
                encoding='utf-8'
            )):
                total_rows += len(chunk)
                # Filter for LOINC source AND ensure the CODE contains a hyphen
                loinc_chunk = chunk[(chunk['SAB'] == 'LNC') & (chunk['CODE'].str.contains('-'))].copy()
                loinc_rows += len(loinc_chunk)

                if loinc_chunk.empty:
                    continue

                # Apply filtering based on config
                if not self.include_obsolete:
                    loinc_chunk = loinc_chunk[loinc_chunk['TS'] == 'P']
                if not self.include_suppressible:
                    loinc_chunk = loinc_chunk[loinc_chunk['SUPPRESS'] == 'N']
                if self.preferred_language:
                    loinc_chunk = loinc_chunk[loinc_chunk['LAT'] == self.preferred_language]

                # Extract mappings
                for _, row in loinc_chunk.iterrows():
                    loinc_code = row['CODE']
                    cui = row['CUI']
                    if loinc_code and cui:
                        loinc_mappings[loinc_code] = cui

                if chunk_num % 10 == 0:
                    self.logger.info(f"Processed {total_rows:,} rows, found {len(loinc_mappings):,} LOINC mappings")

            end_time = datetime.now()
            processing_time = (end_time - start_time).total_seconds()

            self.logger.info(f"Extraction completed in {processing_time:.1f} seconds. Total unique mappings: {len(loinc_mappings):,}")
            self._save_mappings(loinc_mappings)
            return loinc_mappings
        except Exception as e:
            self.logger.error(f"Error extracting from MRCONSO.RRF: {e}")
            raise

    def _save_mappings(self, mappings: Dict[str, str]):
        """Save mappings to output file."""
        if not mappings:
            self.logger.warning("No mappings to save")
            return

        # Construct the output path using the configured output_directory
        simple_output_path = self.output_directory / "loinc_cui_simple_lookup.json"
        with open(simple_output_path, 'w') as f:
            json.dump(mappings, f, indent=2)
        self.logger.info(f"Saved simple lookup to: {simple_output_path}")

# ... (rest of the code for SimpleLOINCMapper and setup_umls_cache) ...

# ============================================================================
# Enhanced SimpleLOINC Mapper with Multi-Level Caching
# ============================================================================

class SimpleLOINCMapper:
    """Enhanced LOINC to CUI mapper with multi-level caching and config support."""

    def __init__(self,
                 api_key: Optional[str] = None,
                 base_url: str = "https://uts-ws.nlm.nih.gov/rest",
                 rate_limit: float = 0.2,
                 api_cache_file: Optional[Path] = None,
                 config_path: Optional[str] = None,
                 umls_cache_file: Optional[Path] = None):
        """
        Initialize the mapper with multi-level caching.
        """
        self.logger = logging.getLogger(__name__)

        self.config = {}
        if config_path:
            self.config = load_config(config_path)

        self.api_key = api_key or self.config.get("api_key")
        self.base_url = base_url
        self.rate_limit = rate_limit if rate_limit != 0.2 else self.config.get("rate_limit_delay", 0.2)

        settings = self.config.get("settings", {})
        self.max_retries = settings.get("max_retries", 3)

        self.session = requests.Session()
        self.session.headers.update({'User-Agent': 'SimpleLOINCMapper/2.0', 'Accept': 'application/json'})

        # Use the output directory from config for the API cache file as well
        output_dir = Path(self.config.get("output_directory", "./")) # Ensure default points to current dir
        output_dir.mkdir(exist_ok=True)
        self.api_cache_file = api_cache_file or output_dir / f"loinc_cui_api_cache_{datetime.now().strftime('%Y-%m-%d')}.csv"


        self.stats = {'umls_cache_hits': 0, 'api_cache_hits': 0, 'api_calls': 0, 'not_found': 0}

        self.umls_cache = self._load_umls_cache(umls_cache_file)
        self.api_cache = self._load_api_cache()

        if self.umls_cache:
            self.logger.info(f"Initialized with local UMLS cache: {len(self.umls_cache):,} mappings")
        self.logger.info(f"Initialized with API cache: {len(self.api_cache):,} mappings")
        if not self.api_key:
            self.logger.warning("No API key provided. Only cached results will be available.")

    def _load_umls_cache(self, cache_file: Optional[Path]) -> Dict[str, str]:
        """Load local UMLS cache file from the specified path."""
        if cache_file and cache_file.exists():
            try:
                with open(cache_file, 'r') as f:
                    return json.load(f)
            except Exception as e:
                self.logger.warning(f"Error loading UMLS cache from {cache_file}: {e}")
        self.logger.info("No local UMLS cache found or specified. Starting with empty cache.")
        return {}

    def _load_api_cache(self) -> Dict[str, str]:
        """Load API results cache from CSV file."""
        if self.api_cache_file.exists():
            try:
                df = pd.read_csv(self.api_cache_file)
                cache_dict = dict(zip(df['loinc_code'].astype(str), df['cui'].astype(str)))
                self.logger.info(f"Loaded API results cache: {len(cache_dict):,} mappings")
                return cache_dict
            except Exception as e:
                self.logger.warning(f"Error loading API cache from {self.api_cache_file}: {e}. Starting with empty cache.")
        return {}

    def _save_api_cache(self, loinc_code: str, cui: str):
        """Append a new mapping to the API cache file."""
        try:
            df = pd.DataFrame([{'loinc_code': loinc_code, 'cui': cui}])
            df.to_csv(self.api_cache_file, mode='a', header=not self.api_cache_file.exists(), index=False)
        except Exception as e:
            self.logger.warning(f"Error saving to API cache: {e}")

    def search_loinc_code(self, loinc_code: str) -> Optional[Dict]:
        """
        Search for a LOINC code using multi-level caching.
        """
        loinc_code = str(loinc_code).strip()
        if loinc_code in self.umls_cache:
            self.stats['umls_cache_hits'] += 1
            cui = self.umls_cache[loinc_code]
            return {'cui': cui, 'cui_name': '', 'source_ui': loinc_code, 'source_name': 'LNC', 'mapping_method': 'umls_local_cache'}
        if loinc_code in self.api_cache:
            self.stats['api_cache_hits'] += 1
            cui = self.api_cache[loinc_code]
            return {'cui': cui, 'cui_name': '', 'source_ui': loinc_code, 'source_name': 'LNC', 'mapping_method': 'api_cache'}
        if not self.api_key:
            self.stats['not_found'] += 1
            self.logger.debug(f"No API key available for {loinc_code}")
            return None
        strategies = [self._search_exact_in_loinc, self._search_general_in_loinc, self._search_unrestricted]
        for strategy in strategies:
            result = strategy(loinc_code)
            if result:
                self.stats['api_calls'] += 1
                self.logger.debug(f"Successfully mapped {loinc_code} using {strategy.__name__}")
                cui = result.get('cui')
                if cui:
                    self._save_api_cache(loinc_code, cui)
                    self.api_cache[loinc_code] = cui
                return result
        self.stats['not_found'] += 1
        return None

    def _search_exact_in_loinc(self, loinc_code: str) -> Optional[Dict]:
        """Search for exact match in LOINC source."""
        params = {'string': loinc_code, 'apiKey': self.api_key, 'sabs': 'LNC', 'searchType': 'exact', 'returnIdType': 'code'}
        return self._execute_search(loinc_code, params, "exact_loinc")

    def _search_general_in_loinc(self, loinc_code: str) -> Optional[Dict]:
        """Search with general terms in LOINC source."""
        params = {'string': loinc_code, 'apiKey': self.api_key, 'sabs': 'LNC', 'searchType': 'words'}
        return self._execute_search(loinc_code, params, "general_loinc")

    def _search_unrestricted(self, loinc_code: str) -> Optional[Dict]:
        """Search without source restriction."""
        params = {'string': loinc_code, 'apiKey': self.api_key, 'searchType': 'exact'}
        return self._execute_search(loinc_code, params, "unrestricted")

    def _execute_search(self, loinc_code: str, params: Dict, method: str) -> Optional[Dict]:
        """Execute API search with given parameters and retry logic."""
        url = f"{self.base_url}/search/current"
        for attempt in range(self.max_retries):
            try:
                response = self.session.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    data = response.json()
                    results = data.get('result', {}).get('results', [])
                    if results:
                        return self._extract_cui_info(loinc_code, results[0], method)
                    return None
                self.logger.warning(f"API request failed for {loinc_code} (attempt {attempt + 1}): {response.status_code}. Retrying...")
                time.sleep(self.rate_limit * (attempt + 1))
            except Exception as e:
                self.logger.error(f"Error executing API search for {loinc_code} (attempt {attempt + 1}): {str(e)}. Retrying...")
                time.sleep(self.rate_limit * (attempt + 1))
        self.logger.error(f"API search failed for {loinc_code} after {self.max_retries} attempts.")
        return None

    def _extract_cui_info(self, loinc_code: str, result: Dict, method: str) -> Optional[Dict]:
        """Extract CUI information from API result."""
        try:
            uri = result.get('uri', '')
            if '/CUI/' not in uri: return None
            cui = uri.split('/CUI/')[-1].split('/')[0].split('?')[0]
            if not cui: return None
            source_ui = result.get('ui', '')
            if source_ui.startswith('MTH'):
                self.logger.debug(f"Skipping MTH code from API result: {source_ui}")
                return None
            return {'cui': cui, 'cui_name': result.get('name', ''), 'source_ui': source_ui, 'source_name': result.get('rootSource', ''), 'mapping_method': method}
        except Exception as e:
            self.logger.error(f"Error extracting CUI for {loinc_code}: {str(e)}")
            return None

    def get_cache_info(self) -> Dict[str, int]:
        """Get information about cache sizes."""
        return {'umls_cache_size': len(self.umls_cache), 'api_cache_size': len(self.api_cache), 'total_cache_size': len(self.umls_cache) + len(self.api_cache)}

# New function to handle setup and initialization
# ============================================================================

def setup_umls_cache(rrf_file_path: str, cache_file_path: str, config_file: str) -> SimpleLOINCMapper:
    """
    Handles the complete setup and initialization of the UMLS cache.
    """
    config = load_config(config_file)
    setup_logging(config.get('log_level', 'INFO'))

    rrf_path = Path(rrf_file_path)
    # Use the provided cache_file_path directly
    cache_path = Path(cache_file_path)

    # Create the directory for the cache file if it doesn't exist
    cache_path.parent.mkdir(parents=True, exist_ok=True)


    print(f"\n🔍 Checking for local UMLS cache...")
    print(f"   Cache file path: {cache_path}")
    print(f"   RRF file path: {rrf_path}")

    if not cache_path.exists():
        print(f"❌ Local UMLS cache not found")
        if rrf_path.exists():
            print(f"✅ Found MRCONSO.RRF file. Creating local cache from RRF file...")
            try:
                # Use the output_directory from the loaded config for the extractor
                extractor_config = config.copy()
                extractor_config["output_directory"] = str(cache_path.parent) # Set extractor output to the cache file's directory
                extractor = UMLSLoincExtractor(config=extractor_config)
                extractor.extract_from_file(rrf_path)
                print(f"✅ Successfully created UMLS cache.")
            except Exception as e:
                print(f"❌ Error creating UMLS cache: {e}. Will proceed with API-only mapping.")
        else:
            print(f"❌ MRCONSO.RRF file not found at {rrf_path}.")
            print(f"   Will proceed with API-only mapping.")
    else:
        print(f"✅ Found existing UMLS cache at {cache_path}")

    print(f"\n🔧 Initializing enhanced SimpleLOINCMapper...")
    try:
        mapper = SimpleLOINCMapper(
            config_path=config_file,
            umls_cache_file=cache_path # Pass the correct cache file path
        )
        cache_info = mapper.get_cache_info()
        print(f"✅ Successfully initialized enhanced SimpleLOINCMapper")
        print(f"📊 Cache Status:")
        print(f"   Local UMLS cache: {cache_info['umls_cache_size']:,} mappings")
        print(f"   API results cache: {cache_info['api_cache_size']:,} mappings")
        print(f"   Total cached mappings: {cache_info['total_cache_size']:,}")

        if cache_info['umls_cache_size'] > 0:
            print(f"🚀 Ready for high-speed UMLS mapping!")
            print(f"   Expected speed: ~660x faster than pure API calls")
        else:
            print(f"⚠️  No local cache available - will use API calls only")

        return mapper
    except Exception as e:
        print(f"❌ Error initializing mapper: {e}")
        raise

# Call the setup function to perform the UMLS cache initialization.
# This will either load an existing cache or create a new one from the RRF file.

# Define paths for the RRF file, the cache, and the config.
rrf_file_path = "Input/umls-2025AA-mrconso/2025AA/META/MRCONSO.RRF"
# Change the cache file path to the content directory
cache_file_path = "loinc_cui_simple_lookup.json"
config_file = "UMLS_API_config.json"

# The 'mapper' object is now ready to be used in the next phase for mapping LOINC concepts to CUIs.
mapper = setup_umls_cache(rrf_file_path, cache_file_path, config_file)

# The `mapper` object is now initialized and ready for use in the next cell.

In [ ]:
# UMLS Enhancement to LOINC Concepts

import pandas as pd
import time
import os
import json
from pathlib import Path

# ============================================================================
# Phase 5: UMLS Enhancement to LOINC Concepts
# ============================================================================

print("🚀 Starting Enhanced LOINC to UMLS CUI Mapping...")
print("=" * 60)

# Verify mapper is available and get cache info
try:
    mapper = SimpleLOINCMapper(
        api_key=None,  # This change disables all API calls
        umls_cache_file=Path("loinc_cui_simple_lookup.json")
    )

    cache_info = mapper.get_cache_info()
    print("✅ Using pre-initialized SimpleLOINCMapper with no API calls")
    print(f"📊 Cache Status:")
    print(f"   Local UMLS cache: {cache_info['umls_cache_size']:,} mappings")
    print(f"   API results cache: {cache_info['api_cache_size']:,} mappings")
    print(f"   Total cached mappings: {cache_info['total_cache_size']:,}")

except NameError:
    print("❌ Mapper or related classes not found! Please run the UMLS Setup cell first.")
    raise
except Exception as e:
    print(f"❌ Error accessing mapper: {e}")
    raise

# Filter concepts to exclude Answer Lists
print("\n📋 Preparing LOINC codes for mapping...")
loinc_codes_all = concepts_with_hierarchy_df['id'].unique()
filtered_concepts_df = concepts_with_hierarchy_df[concepts_with_hierarchy_df['concept_class'] != 'Answer List']
loinc_codes_to_map = filtered_concepts_df['id'].unique()

print(f"   Total unique concept IDs: {len(loinc_codes_all):,}")
print(f"   LOINC codes to map (excluding Answer Lists): {len(loinc_codes_to_map):,}")
print(f"   Answer Lists excluded from mapping: {len(loinc_codes_all) - len(loinc_codes_to_map):,}")

# Process LOINC codes using a loop
print(f"\n📄 Processing {len(loinc_codes_to_map):,} LOINC codes...")
print("   Using SimpleLOINCMapper...")

start_time = time.time()

successful_mappings = []
failed_codes = []
processed_count = 0

# Loop through each LOINC code and perform the mapping
for loinc_code in loinc_codes_to_map:
    result = mapper.search_loinc_code(loinc_code)

    if result and 'cui' in result:
        successful_mappings.append(result)
    else:
        failed_codes.append(loinc_code)

    processed_count += 1
    if processed_count % 1000 == 0:
        elapsed_time = time.time() - start_time
        print(f"   Processed {processed_count:,} codes in {elapsed_time:.1f} seconds. Mappings found: {len(successful_mappings):,}")

end_time = time.time()
processing_time = end_time - start_time

print(f"\n📊 Mapping Results Summary:")
print(f"   Total codes processed: {len(loinc_codes_to_map):,}")
print(f"   Successful mappings: {len(successful_mappings):,}")
print(f"   Failed mappings: {len(failed_codes):,}")
print(f"   Success rate: {(len(successful_mappings) / len(loinc_codes_to_map) * 100):.1f}%")
print(f"   Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")

# Show performance statistics from the enhanced mapper
stats = mapper.stats
print(f"\n🚀 Performance Statistics:")
print(f"   UMLS cache hits: {stats['umls_cache_hits']:,}")
print(f"   API cache hits: {stats['api_cache_hits']:,}")
print(f"   Fresh API calls: {stats['api_calls']:,}")
print(f"   Not found: {stats['not_found']:,}")

if stats['umls_cache_hits'] > 0:
    cache_hit_rate = (stats['umls_cache_hits'] + stats['api_cache_hits']) / len(loinc_codes_to_map) * 100
    print(f"   Cache hit rate: {cache_hit_rate:.1f}%")
    if processing_time < 300:
        print(f"   🚀 High-speed processing achieved! (~{(len(loinc_codes_to_map)/processing_time):.0f} codes/second)")

# Create DataFrame from successful mappings
print(f"\n📋 Creating mapping DataFrame...")
loinc_cui_df = pd.DataFrame(successful_mappings)
print(f"   Successfully created DataFrame with {len(loinc_cui_df)} rows")

# Prepare for merge
if not loinc_cui_df.empty:
    loinc_cui_df.rename(columns={'source_ui': 'id', 'cui': 'cui_from_mapping'}, inplace=True)
    loinc_cui_df = loinc_cui_df[['id', 'cui_from_mapping']].copy()

    # Merge with the main concepts DataFrame
    print(f"\n📄 Merging UMLS mappings with concept data...")
    merged_loinc_cui_df = pd.merge(
        concepts_with_hierarchy_df,
        loinc_cui_df,
        on='id',
        how='left'
    )

    # Check for 'external_id' column before attempting to use it
    if 'external_id' not in merged_loinc_cui_df.columns:
        print("   'external_id' column not found in original DataFrame. Creating a new one...")
        merged_loinc_cui_df['external_id'] = None

    # Update 'external_id'
    print("   Updating 'external_id' and creating 'extras.UMLS_CUI' columns...")
    merged_loinc_cui_df['external_id'] = merged_loinc_cui_df['cui_from_mapping'].fillna(merged_loinc_cui_df['external_id'])

    # REVISED: Create the new 'extras.UMLS_CUI' column and populate it directly
    if 'extras.UMLS_CUI' not in merged_loinc_cui_df.columns:
        merged_loinc_cui_df['extras.UMLS_CUI'] = None
    merged_loinc_cui_df['extras.UMLS_CUI'] = merged_loinc_cui_df['cui_from_mapping']

    # Final cleanup of temporary column
    if 'cui_from_mapping' in merged_loinc_cui_df.columns:
        merged_loinc_cui_df.drop(columns=['cui_from_mapping'], inplace=True)

    # Count results
    matched_cui_count = merged_loinc_cui_df['external_id'].notna().sum()
    unmatched_cui_count = merged_loinc_cui_df['external_id'].isna().sum()

    print(f"✅ Merge completed successfully:")
    print(f"   Total concepts in final dataset: {len(merged_loinc_cui_df):,}")
    print(f"   Concepts with UMLS CUI: {matched_cui_count:,}")
    print(f"   Concepts without UMLS CUI: {unmatched_cui_count:,}")
    print(f"   UMLS mapping coverage: {(matched_cui_count / len(merged_loinc_cui_df) * 100):.1f}%")
else:
    print("⚠️  No mappings to merge - using original concepts DataFrame")
    merged_loinc_cui_df = concepts_with_hierarchy_df.copy()

# FINAL VALIDATION: Check that CUI fields are present
print(f"\n🔍 Final CUI validation:")
cui_count = merged_loinc_cui_df['external_id'].notna().sum()
print(f"   external_id field: {cui_count:,} non-null values")

# This check is now for the new column name
cui_extras_count = merged_loinc_cui_df['extras.UMLS_CUI'].notna().sum()
print(f"   extras.UMLS_CUI field: {cui_extras_count:,} non-null values")

print(f"\n✅ Phase 5 UMLS Enhancement Complete!")
print(f"   CUI data successfully integrated into concept records.")
print(f"   Proceed to Phase 6 for final output generation.")
print("=" * 60)

## Phase 6: Output Generation

The final phase is to save all the created OCL objects to a JSON files (.json) in the Bulk Import JSON-lines-like format for import into the OCL system.

In [ ]:
# Create Container Concepts and Assign Parents

def create_container_concepts():
    """
    Creates ROOT and Container concepts for organizing the LOINC hierarchy.
    Returns a list of container concept dictionaries.
    """
    container_concepts = []

    # 1. Create the ROOT concept
    root_concept = {
        'type': 'Concept',
        'id': 'ROOT',
        'concept_class': 'Root',
        'datatype': 'N/A',
        'source': global_source,
        'owner_type': 'Organization',
        'owner': global_organization,
        'retired': False,
        'names': [
            {
                'name': 'LOINC Root Concept',
                'name_type': 'Fully-Specified',
                'locale': 'en-GB',
                'locale_preferred': True
            },
            {
                'name': 'LOINC Root',
                'name_type': 'Short',
                'locale': 'en-GB',
                'locale_preferred': False
            }
        ],
        'descriptions': [
            {
                'description': 'Root concept for the LOINC terminology hierarchy',
                'locale': 'en-GB',
                'description_type': 'Full'
            }
        ],
        'extras': {
            'Code_Type': 'Root Container',
            'Code_in_Source': 'ROOT'
        }
        # Note: ROOT has no parent_concept_urls - it's the top of the hierarchy
    }
    container_concepts.append(root_concept)

    # 2. Define Container concepts for each concept class and the new "Other" container
    container_definitions = {
        'LOINC': {
            'id': 'LOINC_CONTAINER',
            'name': 'LOINC Terms Container',
            'short_name': 'LOINC Terms',
            'description': 'Container for all LOINC laboratory terms and observations',
            'parent_id': 'OTHER'
        },
        'LOINC Part': {
            'id': 'LOINC_PARTS_CONTAINER',
            'name': 'LOINC Parts Container',
            'short_name': 'LOINC Parts',
            'description': 'Container for all LOINC component parts used to build LOINC terms',
            'parent_id': 'OTHER'
        },
        'LOINC Answer': {
            'id': 'LOINC_ANSWERS_CONTAINER',
            'name': 'LOINC Answers Container',
            'short_name': 'LOINC Answers',
            'description': 'Container for all LOINC answer concepts used in answer lists',
            'parent_id': 'OTHER'
        },
        'Answer List': {
            'id': 'ANSWER_LISTS_CONTAINER',
            'name': 'Answer Lists Container',
            'short_name': 'Answer Lists',
            'description': 'Container for all LOINC answer lists and value sets',
            'parent_id': 'OTHER'
        }
    }

    # 3. Create the new "Other" container with ROOT as its parent
    other_container_concept = {
        'type': 'Concept',
        'id': 'OTHER',
        'concept_class': 'Container',
        'datatype': 'N/A',
        'source': global_source,
        'owner_type': 'Organization',
        'owner': global_organization,
        'retired': False,
        'names': [
            {
                'name': 'Other LOINC Containers',
                'name_type': 'Fully-Specified',
                'locale': 'en-GB',
                'locale_preferred': True
            },
            {
                'name': 'Other',
                'name_type': 'Short',
                'locale': 'en-GB',
                'locale_preferred': False
            }
        ],
        'descriptions': [
            {
                'description': 'Container for miscellaneous LOINC containers',
                'locale': 'en-GB',
                'description_type': 'Full'
            }
        ],
        'parent_concept_urls': [f"{CONCEPT_URL_PREFIX}{'ROOT'}/"],
        'extras': {
            'Code_Type': 'Container'
        }
    }
    container_concepts.append(other_container_concept)


    # 4. Create the other Container concepts with "Other" as their parent
    for concept_class, container_info in container_definitions.items():
        container_concept = {
            'type': 'Concept',
            'id': container_info['id'],
            'concept_class': 'Container',
            'datatype': 'N/A',
            'source': global_source,
            'owner_type': 'Organization',
            'owner': global_organization,
            'retired': False,
            'names': [
                {
                    'name': container_info['name'],
                    'name_type': 'Fully-Specified',
                    'locale': 'en-GB',
                    'locale_preferred': True
                },
                {
                    'name': container_info['short_name'],
                    'name_type': 'Short',
                    'locale': 'en-GB',
                    'locale_preferred': False
                }
            ],
            'descriptions': [
                {
                    'description': container_info['description'],
                    'locale': 'en-GB',
                    'description_type': 'Full'
                }
            ],
            'parent_concept_urls': [f"{CONCEPT_URL_PREFIX}{container_info['parent_id']}/"],
            'extras': {
                'Code_Type': 'Container',
                'Code_in_Source': container_info['id'],
                'Container_For': concept_class
            }
        }
        container_concepts.append(container_concept)

    return container_concepts

def assign_orphaned_concepts_to_containers(df):
    """
    Assigns concepts without parents to appropriate Container concepts.
    Updates the parent_concept_urls for orphaned concepts.
    """
    # Mapping of concept classes to their container IDs
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER',
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }

    # --- ADDED CODE START ---
    # Define the list of exception concepts that should be assigned to ROOT
    root_exceptions = {
        'LP29693-6',
        'LP29695-1',
        'LP29696-9',
        'LP7787-7'
    }
    # --- ADDED CODE END ---

    # Create a copy to avoid modifying the original dataframe
    df_updated = df.copy()

    # Find concepts without parent_concept_urls
    orphaned_mask = df_updated['parent_concept_urls'].isnull()
    orphaned_concepts = df_updated[orphaned_mask]

    print(f"Found {len(orphaned_concepts)} orphaned concepts to assign to containers:")

    # Group orphaned concepts by concept_class and show counts
    orphaned_by_class = orphaned_concepts.groupby('concept_class').size()
    for concept_class, count in orphaned_by_class.items():
        container_id = concept_class_to_container.get(concept_class, 'UNKNOWN')
        print(f"  - {concept_class}: {count} concepts → {container_id}")

    # Assign orphaned concepts to containers
    for idx, row in orphaned_concepts.iterrows():
        # --- ADDED CODE START ---
        concept_id = row['id']
        if concept_id in root_exceptions:
            # Assign the exception concept to ROOT
            df_updated.at[idx, 'parent_concept_urls'] = [f"{CONCEPT_URL_PREFIX}{'ROOT'}/"]
            print(f"  - EXCEPTION: {concept_id} assigned directly to ROOT.")
            continue # Move to the next concept
        # --- ADDED CODE END ---

        concept_class = row['concept_class']
        container_id = concept_class_to_container.get(concept_class)

        if container_id:
            # Assign to the appropriate container
            df_updated.at[idx, 'parent_concept_urls'] = [f"{CONCEPT_URL_PREFIX}{container_id}/"]
        else:
            # If no specific container, assign to ROOT
            print(f"Warning: No container defined for concept_class '{concept_class}', assigning to ROOT")
            df_updated.at[idx, 'parent_concept_urls'] = [f"{CONCEPT_URL_PREFIX}{'ROOT'}/"]

    return df_updated

def integrate_container_concepts(concepts_df, container_concepts_list):
    """
    Integrates the container concepts into the main concepts dataframe.
    Checks for existing container concepts to prevent duplicates.
    """
    # Check if container concepts already exist
    existing_container_ids = set()
    for container_concept in container_concepts_list:
        container_id = container_concept['id']
        if container_id in concepts_df['id'].values:
            existing_container_ids.add(container_id)

    if existing_container_ids:
        print(f"ℹ️  Skipping {len(existing_container_ids)} container concepts that already exist: {existing_container_ids}")
        # Filter out existing containers
        new_container_concepts = [c for c in container_concepts_list if c['id'] not in existing_container_ids]
    else:
        new_container_concepts = container_concepts_list

    if not new_container_concepts:
        print("ℹ️  No new container concepts to add.")
        return concepts_df

    # Convert new container concepts to DataFrame
    container_df = pd.DataFrame(new_container_concepts)

    # Ensure all columns exist in both dataframes
    all_columns = set(concepts_df.columns) | set(container_df.columns)

    # Add missing columns to both dataframes
    for col in all_columns:
        if col not in concepts_df.columns:
            concepts_df[col] = None
        if col not in container_df.columns:
            container_df[col] = None

    # Reorder columns to match
    container_df = container_df[concepts_df.columns]

    # Concatenate the dataframes
    integrated_df = pd.concat([concepts_df, container_df], ignore_index=True)

    print(f"ℹ️  Added {len(new_container_concepts)} new container concepts.")

    return integrated_df

# Execute the container creation process
print("=== Creating Container Concepts ===")

existing_root = 'ROOT' in merged_loinc_cui_df['id'].values
existing_containers = any(cid in merged_loinc_cui_df['id'].values
                          for cid in ['LOINC_CONTAINER', 'LOINC_PARTS_CONTAINER',
                                      'LOINC_ANSWERS_CONTAINER', 'ANSWER_LISTS_CONTAINER',
                                      'OTHER'])

if existing_root and existing_containers:
    print("ℹ️  Container concepts already exist. Skipping container creation.")
    print("ℹ️  If you want to recreate containers, restart the kernel and run from the beginning.")
    concepts_with_containers_df = merged_loinc_cui_df.copy()
else:
    # 1. Create container concepts
    container_concepts_list = create_container_concepts()
    print(f"Created {len(container_concepts_list)} container concepts (including ROOT)")

    # 2. Assign orphaned concepts to containers (work with a copy)
    concepts_with_parents_df = assign_orphaned_concepts_to_containers(merged_loinc_cui_df)

    # 3. Integrate container concepts into the main dataframe
    concepts_with_containers_df = integrate_container_concepts(concepts_with_parents_df, container_concepts_list)

    print(f"\nFinal concept count: {len(concepts_with_containers_df)}")
    print(f"Container concepts: {len([c for c in container_concepts_list if c['id'] not in merged_loinc_cui_df['id'].values])}")
    print(f"Original concepts: {len(merged_loinc_cui_df)}")

# 4. Final validation - check for remaining orphaned concepts (excluding ROOT)
remaining_orphaned = concepts_with_containers_df[
    (concepts_with_containers_df['parent_concept_urls'].isnull()) &
    (concepts_with_containers_df['id'] != 'ROOT')
]
if len(remaining_orphaned) == 0:
    print("✅ SUCCESS: All concepts now have parent assignments!")
else:
    print(f"⚠️  WARNING: {len(remaining_orphaned)} concepts still without parents:")
    print(remaining_orphaned[['id', 'concept_class']].head())

# 5. Sort concepts hierarchically so parents come before children
def sort_concepts_hierarchically(df):
    """
    Sorts concepts so that parent concepts appear before their children.
    Uses topological sorting based on parent_concept_urls.
    """
    print("Sorting concepts hierarchically...")

    # Create a copy to work with
    df_sorted = df.copy().reset_index(drop=True)

    # Extract parent IDs from parent_concept_urls
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            # Extract IDs from URLs like "/orgs/Regenstrief/sources/LOINC/concepts/ROOT/"
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    # Extract the concept ID from the URL
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:  # Expected format: orgs/Regenstrief/sources/LOINC/concepts/ID
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_sorted['parent_ids'] = df_sorted['parent_concept_urls'].apply(extract_parent_ids)

    # Topological sort
    sorted_concepts = []
    processed_ids = set()
    remaining_df = df_sorted.copy()

    iteration = 0
    max_iterations = len(df_sorted) + 10  # Safety limit

    while len(remaining_df) > 0 and iteration < max_iterations:
        iteration += 1

        # Find concepts whose parents are already processed (or have no parents)
        ready_concepts = []

        for idx, row in remaining_df.iterrows():
            parent_ids = row['parent_ids']

            # Concept is ready if it has no parents or all parents are already processed
            if not parent_ids or all(pid in processed_ids for pid in parent_ids):
                ready_concepts.append(idx)

        if not ready_concepts:
            # If no concepts are ready, we might have circular dependencies
            # Add remaining concepts in order of their dependency count
            print(f"Warning: Possible circular dependencies detected. Adding remaining {len(remaining_df)} concepts by dependency count.")
            dependency_counts = remaining_df['parent_ids'].apply(lambda x: len([p for p in x if p not in processed_ids]))
            ready_concepts = dependency_counts.sort_values().index.tolist()

        # Add ready concepts to sorted list
        for idx in ready_concepts:
            row = remaining_df.loc[idx]
            sorted_concepts.append(row)
            processed_ids.add(row['id'])

        # Remove processed concepts from remaining
        remaining_df = remaining_df.drop(ready_concepts)

        if iteration % 50 == 0:
            print(f"  Processed {len(sorted_concepts)}/{len(df_sorted)} concepts...")

    # Create the final sorted dataframe
    final_sorted_df = pd.DataFrame(sorted_concepts).reset_index(drop=True)

    # Drop the temporary parent_ids column
    final_sorted_df = final_sorted_df.drop('parent_ids', axis=1)

    print(f"Hierarchical sorting complete. Processed {len(final_sorted_df)} concepts in {iteration} iterations.")

    return final_sorted_df

# Debug code to identify circular dependencies - FIXED VERSION
def debug_circular_dependencies(df):
    """
    Analyzes the concepts dataframe to identify potential circular dependencies.
    """
    print("\n=== DEBUG: Analyzing Circular Dependencies ===")

    # Extract parent IDs from parent_concept_urls (FIXED VERSION)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_debug = df.copy()
    df_debug['parent_ids'] = df_debug['parent_concept_urls'].apply(extract_parent_ids)

    # Create a dictionary of concept -> parents mapping
    concept_parents = {}
    for _, row in df_debug.iterrows():
        concept_id = row['id']
        parent_ids = row['parent_ids']
        concept_parents[concept_id] = parent_ids

    # Find concepts that are involved in cycles
    def find_path_to_concept(start_concept, target_concept, visited=None):
        """Returns the path if target_concept is reachable from start_concept"""
        if visited is None:
            visited = set()

        if start_concept in visited:
            return None  # Already visited, potential cycle

        if start_concept == target_concept:
            return [start_concept]

        visited.add(start_concept)

        # Check all parents of the current concept
        parents = concept_parents.get(start_concept, [])
        for parent in parents:
            if parent in concept_parents:  # Only follow parents that exist as concepts
                path = find_path_to_concept(parent, target_concept, visited.copy())
                if path:
                    return [start_concept] + path

        return None

    # Check for cycles
    cycles_found = []
    concepts_in_cycles = set()

    for concept_id in concept_parents.keys():
        # Check if this concept can reach itself through its parents
        parents = concept_parents.get(concept_id, [])
        for parent in parents:
            if parent in concept_parents:
                cycle_path = find_path_to_concept(parent, concept_id)
                if cycle_path:
                    full_cycle = [concept_id] + cycle_path
                    cycles_found.append(full_cycle)
                    concepts_in_cycles.update(full_cycle)

    if cycles_found:
        print(f"🚨 Found {len(cycles_found)} circular dependency cycles:")
        for i, cycle in enumerate(cycles_found, 1):
            print(f"  Cycle {i}: {' → '.join(cycle)} → {cycle[0]}")

    # Identify concepts that might be causing the sorting issues
    problem_concepts = []
    all_concept_ids = set(concept_parents.keys())

    # Find concepts whose parents don't exist as concepts
    for concept_id, parent_ids in concept_parents.items():
        missing_parents = [p for p in parent_ids if p not in all_concept_ids]
        if missing_parents:
            problem_concepts.append({
                'concept_id': concept_id,
                'issue': 'missing_parents',
                'details': missing_parents
            })

    # Find concepts with complex parent relationships
    multi_parent_concepts = []
    for concept_id, parent_ids in concept_parents.items():
        if len(parent_ids) > 1:
            multi_parent_concepts.append({
                'concept_id': concept_id,
                'parent_count': len(parent_ids),
                'parents': parent_ids
            })

    print(f"\n📊 Dependency Analysis Summary:")
    print(f"   Total concepts: {len(concept_parents)}")
    print(f"   Concepts in cycles: {len(concepts_in_cycles)}")
    print(f"   Concepts with missing parents: {len(problem_concepts)}")
    print(f"   Concepts with multiple parents: {len(multi_parent_concepts)}")

    if problem_concepts:
        print(f"\n⚠️  Concepts with missing parents:")
        for problem in problem_concepts[:5]:  # Show first 5
            print(f"   {problem['concept_id']} → missing: {problem['details']}")
        if len(problem_concepts) > 5:
            print(f"   ... and {len(problem_concepts) - 5} more")

    if multi_parent_concepts:
        print(f"\n📋 Concepts with multiple parents:")
        for concept in multi_parent_concepts[:5]:  # Show first 5
            print(f"   {concept['concept_id']} → parents: {concept['parents']}")
        if len(multi_parent_concepts) > 5:
            print(f"   ... and {len(multi_parent_concepts) - 5} more")

    # Show the specific concepts that are likely causing the sorting warning
    if concepts_in_cycles:
        print(f"\n🔍 Concepts likely causing the sorting warning:")
        cycle_concept_details = df_debug[df_debug['id'].isin(concepts_in_cycles)][['id', 'concept_class', 'parent_ids']]
        print(cycle_concept_details.to_string(index=False))

    print("=== End Debug Analysis ===\n")

    return concepts_in_cycles, problem_concepts, multi_parent_concepts

# Run the debug analysis
debug_results = debug_circular_dependencies(concepts_with_containers_df)

# Fix missing parent references by reassigning to containers
def fix_missing_parent_references(df):
    """
    Identifies concepts with missing parent references and reassigns them to appropriate containers.
    """
    print("\n=== FIXING: Missing Parent References ===")

    # Get all concept IDs that exist in our dataset
    existing_concept_ids = set(df['id'].values)

    # Container mapping
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER',
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }

    # Extract parent IDs function (same logic as above)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_fixed = df.copy()
    concepts_fixed = []

    for idx, row in df_fixed.iterrows():
            concept_id = row['id']
            parent_urls = row['parent_concept_urls']
            concept_class = row['concept_class']

            # Use extract_parent_ids to safely handle all parent_urls formats
            parent_ids = extract_parent_ids(parent_urls)

            if parent_ids:  # Only proceed if we have parent IDs to check
                missing_parents = [pid for pid in parent_ids if pid not in existing_concept_ids]

                if missing_parents:
                    # This concept has missing parent references
                    # Reassign to appropriate container
                    container_id = concept_class_to_container.get(concept_class, 'ROOT')
                    new_parent_url = f"{CONCEPT_URL_PREFIX}{container_id}/"

                    # Update the parent_concept_urls
                    df_fixed.at[idx, 'parent_concept_urls'] = [new_parent_url]

                    concepts_fixed.append({
                        'concept_id': concept_id,
                        'concept_class': concept_class,
                        'missing_parents': missing_parents,
                        'new_parent': container_id
                    })

    print(f"✅ Fixed {len(concepts_fixed)} concepts with missing parent references")
    print("=== End Fix ===\n")

    return df_fixed, concepts_fixed

# Apply the fix
concepts_with_containers_df_fixed, fixed_concepts = fix_missing_parent_references(concepts_with_containers_df)

# Update the variable name for clarity
concepts_with_containers_df = concepts_with_containers_df_fixed

# Apply hierarchical sorting
final_concepts_with_hierarchy_df = sort_concepts_hierarchically(concepts_with_containers_df)

# 6. Update the global dataframe with the final result
merged_loinc_cui_df = final_concepts_with_hierarchy_df.copy()

print("=== Container Concepts Creation Complete ===")

In [ ]:
# Quick cleanup of dfs before output
# REPLACE THE CLEANUP SECTION IN PHASE 6 WITH THIS FIXED VERSION

def safe_na_to_none(value):
    """
    Safely converts NaN values to None, handling both scalar and array-like values.
    FIXED: Preserves actual CUI values (like "C0000097") that might look like they need conversion.
    """
    # Handle None explicitly
    if value is None:
        return None

    # FIXED: For CUI fields, preserve string values even if they might trigger false positives
    if isinstance(value, str) and value.strip():
        # Don't convert valid CUI strings or other meaningful strings
        return value

    # Handle scalar values (numbers, single values)
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        try:
            return None if pd.isna(value) else value
        except (TypeError, ValueError):
            # If pd.isna() fails for any reason, return the original value
            return value

    # Handle array-like values (lists, arrays, Series)
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return None
        # Check if all elements are NaN
        try:
            if all(pd.isna(v) for v in value):
                return None
            else:
                return value
        except (TypeError, ValueError):
            return value

    elif isinstance(value, pd.Series):
        if value.empty or value.dropna().empty:
            return None
        else:
            return value

    elif isinstance(value, np.ndarray):
        if value.size == 0:
            return None
        try:
            if np.all(pd.isna(value)):
                return None
            else:
                return value
        except (TypeError, ValueError):
            return value

    return value

def is_valid_value(value):
    """
    Checks if a value is not NaN, None, an empty list/dict, or an empty string.
    FIXED: Preserves CUI values and other important strings.
    """
    if value is None:
        return False

    # Handle empty string specifically, but preserve meaningful strings like CUIs
    if isinstance(value, str):
        return value.strip() != ""  # Keep any non-empty string after stripping whitespace

    # Handle scalar values first
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        return not pd.isna(value)

    # Handle non-scalar values (lists, Series, etc.)
    if isinstance(value, pd.Series):
        return not value.dropna().empty
    elif isinstance(value, (list, np.ndarray)):
        return any(pd.notna(v) for v in value)

    # For other types, check if they are "empty"
    if hasattr(value, '__len__') and len(value) == 0:
        return False

    return True

# FIXED: More careful cleanup for CUI-containing DataFrames
print("🧹 Applying enhanced cleanup with CUI preservation...")

# Before cleanup, check current CUI status
if 'external_id' in merged_loinc_cui_df.columns:
    cui_before = merged_loinc_cui_df['external_id'].notna().sum()
    print(f"   external_id before cleanup: {cui_before:,} values")

if 'extras.UMLS_CUI' in merged_loinc_cui_df.columns:
    cui_extras_before = merged_loinc_cui_df['extras.UMLS_CUI'].notna().sum()
    print(f"   extras.UMLS_CUI before cleanup: {cui_extras_before:,} values")

# Apply cleanup to non-CUI extras columns first
print("   Cleaning non-CUI extras columns...")
cui_related_columns = ['external_id', 'extras.UMLS_CUI']
for col in merged_loinc_cui_df.columns:
    if col.startswith('extras.') and col not in cui_related_columns:
        merged_loinc_cui_df[col] = merged_loinc_cui_df[col].apply(safe_na_to_none)

# Special handling for CUI columns - only clean if truly empty/null
print("   Preserving CUI fields during cleanup...")
for cui_col in cui_related_columns:
    if cui_col in merged_loinc_cui_df.columns:
        # Only set to None if the value is actually NaN/null, not if it's a valid string
        mask = merged_loinc_cui_df[cui_col].notna()
        print(f"   Preserving {mask.sum():,} values in {cui_col}")

# After cleanup, verify CUI status
if 'external_id' in merged_loinc_cui_df.columns:
    cui_after = merged_loinc_cui_df['external_id'].notna().sum()
    print(f"   external_id after cleanup: {cui_after:,} values")
    if cui_after != cui_before:
        print(f"   ⚠️  WARNING: Lost {cui_before - cui_after} external_id values during cleanup!")

if 'extras.UMLS_CUI' in merged_loinc_cui_df.columns:
    cui_extras_after = merged_loinc_cui_df['extras.UMLS_CUI'].notna().sum()
    print(f"   extras.UMLS_CUI after cleanup: {cui_extras_after:,} values")
    if cui_extras_after != cui_extras_before:
        print(f"   ⚠️  WARNING: Lost {cui_extras_before - cui_extras_after} extras.UMLS_CUI values during cleanup!")

# Continue with existing cleanup...
merged_loinc_cui_df['extras.VersionLastChanged'] = merged_loinc_cui_df['extras.VersionLastChanged'].astype(str).replace('nan', '')
merged_loinc_cui_df.drop(columns=["parent_concept","match-id"], inplace=True, errors='ignore')

print("✅ Enhanced cleanup completed with CUI preservation")


In [ ]:
# Create Container Concepts and Assign Parents

def create_container_concepts():
    """
    Creates ROOT and Container concepts for organizing the LOINC hierarchy.
    Returns a list of container concept dictionaries.
    """
    container_concepts = []

    # 1. Create the ROOT concept
    root_concept = {
        'type': 'Concept',
        'id': 'ROOT',
        'concept_class': 'Root',
        'datatype': 'N/A',
        'source': global_source,
        'owner_type': 'Organization',
        'owner': global_organization,
        'retired': False,
        'names': [
            {
                'name': 'LOINC Root Concept',
                'name_type': 'Fully-Specified',
                'locale': 'en-GB',
                'locale_preferred': True
            },
            {
                'name': 'LOINC Root',
                'name_type': 'Short',
                'locale': 'en-GB',
                'locale_preferred': False
            }
        ],
        'descriptions': [
            {
                'description': 'Root concept for the LOINC terminology hierarchy',
                'locale': 'en-GB',
                'description_type': 'Full'
            }
        ],
        'extras': {
            'Code_Type': 'Root Container',
            'Code_in_Source': 'ROOT'
        }
        # Note: ROOT has no parent_concept_urls - it's the top of the hierarchy
    }
    container_concepts.append(root_concept)

    # 2. Define Container concepts for each concept class and the new "Other" container
    container_definitions = {
        'LOINC': {
            'id': 'LOINC_CONTAINER',
            'name': 'LOINC Terms Container',
            'short_name': 'LOINC Terms',
            'description': 'Container for all LOINC laboratory terms and observations',
            'parent_id': 'OTHER'
        },
        'LOINC Part': {
            'id': 'LOINC_PARTS_CONTAINER',
            'name': 'LOINC Parts Container',
            'short_name': 'LOINC Parts',
            'description': 'Container for all LOINC component parts used to build LOINC terms',
            'parent_id': 'OTHER'
        },
        'LOINC Answer': {
            'id': 'LOINC_ANSWERS_CONTAINER',
            'name': 'LOINC Answers Container',
            'short_name': 'LOINC Answers',
            'description': 'Container for all LOINC answer concepts used in answer lists',
            'parent_id': 'OTHER'
        },
        'Answer List': {
            'id': 'ANSWER_LISTS_CONTAINER',
            'name': 'Answer Lists Container',
            'short_name': 'Answer Lists',
            'description': 'Container for all LOINC answer lists and value sets',
            'parent_id': 'OTHER'
        }
    }

    # 3. Create the new "Other" container with ROOT as its parent
    other_container_concept = {
        'type': 'Concept',
        'id': 'OTHER',
        'concept_class': 'Container',
        'datatype': 'N/A',
        'source': global_source,
        'owner_type': 'Organization',
        'owner': global_organization,
        'retired': False,
        'names': [
            {
                'name': 'Other LOINC Containers',
                'name_type': 'Fully-Specified',
                'locale': 'en-GB',
                'locale_preferred': True
            },
            {
                'name': 'Other',
                'name_type': 'Short',
                'locale': 'en-GB',
                'locale_preferred': False
            }
        ],
        'descriptions': [
            {
                'description': 'Container for miscellaneous LOINC containers',
                'locale': 'en-GB',
                'description_type': 'Full'
            }
        ],
        'parent_concept_urls': [f"{CONCEPT_URL_PREFIX}{'ROOT'}/"],
        'extras': {
            'Code_Type': 'Container'
        }
    }
    container_concepts.append(other_container_concept)


    # 4. Create the other Container concepts with "Other" as their parent
    for concept_class, container_info in container_definitions.items():
        container_concept = {
            'type': 'Concept',
            'id': container_info['id'],
            'concept_class': 'Container',
            'datatype': 'N/A',
            'source': global_source,
            'owner_type': 'Organization',
            'owner': global_organization,
            'retired': False,
            'names': [
                {
                    'name': container_info['name'],
                    'name_type': 'Fully-Specified',
                    'locale': 'en-GB',
                    'locale_preferred': True
                },
                {
                    'name': container_info['short_name'],
                    'locale': 'en-GB',
                    'locale_preferred': False,
                    'name_type': 'Short', # Corrected key order
                }
            ],
            'descriptions': [
                {
                    'description': container_info['description'],
                    'locale': 'en-GB',
                    'description_type': 'Full'
                }
            ],
            'parent_concept_urls': [f"{CONCEPT_URL_PREFIX}{container_info['parent_id']}/"],
            'extras': {
                'Code_Type': 'Container',
                'Code_in_Source': container_info['id'],
                'Container_For': concept_class
            }
        }
        container_concepts.append(container_concept)

    return container_concepts

def assign_orphaned_concepts_to_containers(df):
    """
    Assigns concepts without parents to appropriate Container concepts.
    Updates the parent_concept_urls for orphaned concepts, preserving existing valid parents.
    Explicitly excludes 'ROOT' concept from assignment logic.
    """
    # Mapping of concept classes to their container IDs
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER',
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }

    # Define the list of exception concepts that should be assigned to ROOT
    root_exceptions = {
        'LP29693-6',
        'LP29695-1',
        'LP29696-9',
        'LP7787-7'
    }
    root_parent_url = f"{CONCEPT_URL_PREFIX}{'ROOT'}/"

    # Create a copy to avoid modifying the original dataframe
    df_updated = df.copy()

    # Process each concept to check its parent assignment
    orphaned_count = 0
    container_assigned_count = 0
    exception_assigned_count = 0

    for idx, row in df_updated.iterrows():
        concept_id = row['id']
        current_parents = row.get('parent_concept_urls')
        concept_class = row['concept_class']

        # *** ADDED CHECK: Skip the ROOT concept itself ***
        if concept_id == 'ROOT':
             # Ensure ROOT has no parents if it somehow got one
             df_updated.at[idx, 'parent_concept_urls'] = None
             continue

        # Check if the concept is one of the ROOT exceptions
        if concept_id in root_exceptions:
            # Explicitly set parent to ROOT, overwriting any other parents
            df_updated.at[idx, 'parent_concept_urls'] = [root_parent_url]
            exception_assigned_count += 1
            # print(f"  - EXCEPTION: {concept_id} assigned directly to ROOT.") # Optional debug print
            continue # Move to the next concept


        # Check if the concept already has valid parent(s) assigned
        # A valid parent_concept_urls should be a non-empty list of strings
        has_valid_parent = (
            isinstance(current_parents, list) and
            len(current_parents) > 0 and
            all(isinstance(url, str) and url.strip() for url in current_parents)
        )

        if not has_valid_parent:
            # This concept is orphaned or has invalid parent entries
            orphaned_count += 1

            container_id = concept_class_to_container.get(concept_class)

            if container_id:
                # Assign to the appropriate container
                df_updated.at[idx, 'parent_concept_urls'] = [f"{CONCEPT_URL_PREFIX}{container_id}/"]
                container_assigned_count += 1
            else:
                # If no specific container, assign to ROOT
                # This case should be rare if containers cover all classes, but good fallback
                print(f"Warning: No container defined for concept_class '{concept_class}' ({concept_id}), assigning to ROOT")
                df_updated.at[idx, 'parent_concept_urls'] = [root_parent_url]
                container_assigned_count += 1 # Count as assigned to a container (ROOT is a container type)


    print(f"Found {orphaned_count} concepts initially without valid parent assignments (excluding ROOT).")
    print(f"Assigned {container_assigned_count} concepts to containers.")
    print(f"Explicitly assigned {exception_assigned_count} exception concepts to ROOT.")

    return df_updated

def integrate_container_concepts(concepts_df, container_concepts_list):
    """
    Integrates the container concepts into the main concepts dataframe.
    Checks for existing container concepts to prevent duplicates.
    """
    # Check if container concepts already exist
    existing_container_ids = set()
    for container_concept in container_concepts_list:
        container_id = container_concept['id']
        if container_id in concepts_df['id'].values:
            existing_container_ids.add(container_id)

    if existing_container_ids:
        print(f"ℹ️  Skipping {len(existing_container_ids)} container concepts that already exist: {existing_container_ids}")
        # Filter out existing containers
        new_container_concepts = [c for c in container_concepts_list if c['id'] not in existing_container_ids]
    else:
        new_container_concepts = container_concepts_list

    if not new_container_concepts:
        print("ℹ️  No new container concepts to add.")
        return concepts_df

    # Convert new container concepts to DataFrame
    container_df = pd.DataFrame(new_container_concepts)

    # Ensure all columns exist in both dataframes
    all_columns = set(concepts_df.columns) | set(container_df.columns)

    # Add missing columns to both dataframes
    for col in all_columns:
        if col not in concepts_df.columns:
            concepts_df[col] = None
        if col not in container_df.columns:
            container_df[col] = None

    # Reorder columns to match
    container_df = container_df[concepts_df.columns]

    # Concatenate the dataframes
    integrated_df = pd.concat([concepts_df, container_df], ignore_index=True)

    print(f"ℹ️  Added {len(new_container_concepts)} new container concepts.")

    return integrated_df

# Execute the container creation process
print("=== Creating Container Concepts ===")

# Check if container concepts already exist in the *current* state of merged_loinc_cui_df
# This handles cases where the cell might be re-run after containers were added
existing_root = 'ROOT' in merged_loinc_cui_df.get('id', pd.Series()).values
existing_containers = any(cid in merged_loinc_cui_df.get('id', pd.Series()).values
                          for cid in ['LOINC_CONTAINER', 'LOINC_PARTS_CONTAINER',
                                      'LOINC_ANSWERS_CONTAINER', 'ANSWER_LISTS_CONTAINER',
                                      'OTHER'])

if existing_root or existing_containers: # Check if *any* container/root exists
    print("ℹ️  Container concepts or ROOT already exist. Skipping container creation and integration.")
    print("ℹ️  Proceeding to re-assign parents based on existing concepts and containers.")
    # If containers exist, we still need to run assign_orphaned_concepts_to_containers
    # to fix any concepts that might have lost their parents or had invalid ones.
    concepts_with_containers_df = assign_orphaned_concepts_to_containers(merged_loinc_cui_df)
else:
    # 1. Create container concepts
    container_concepts_list = create_container_concepts()
    print(f"Created {len(container_concepts_list)} container concepts (including ROOT)")

    # 2. Assign orphaned concepts to containers (work with a copy)
    concepts_with_parents_df = assign_orphaned_concepts_to_containers(merged_loinc_cui_df)

    # 3. Integrate container concepts into the main dataframe
    concepts_with_containers_df = integrate_container_concepts(concepts_with_parents_df, container_concepts_list)

    print(f"\nFinal concept count: {len(concepts_with_containers_df)}")
    # Calculate how many containers were actually added in this run
    added_container_count = len([c for c in container_concepts_list if c['id'] not in merged_loinc_cui_df.get('id', pd.Series()).values])
    print(f"Container concepts added in this run: {added_container_count}")
    print(f"Original concepts count before adding containers: {len(merged_loinc_cui_df)}")


# 4. Final validation - check for remaining orphaned concepts (excluding ROOT and Containers)
remaining_orphaned = concepts_with_containers_df[
    (concepts_with_containers_df['parent_concept_urls'].isnull()) &
    (concepts_with_containers_df['id'] != 'ROOT') &
    (~concepts_with_containers_df['concept_class'].isin(['Root', 'Container'])) # Exclude containers themselves
]

if len(remaining_orphaned) == 0:
    print("✅ SUCCESS: All non-container concepts now have parent assignments!")
else:
    print(f"⚠️  WARNING: {len(remaining_orphaned)} non-container concepts still without parents:")
    # print(remaining_orphaned[['id', 'concept_class']].head()) # Optional: print head of orphaned concepts

# 5. Sort concepts hierarchically so parents come before children
def sort_concepts_hierarchically(df):
    """
    Sorts concepts so that parent concepts appear before their children.
    Uses topological sorting based on parent_concept_urls.
    """
    print("Sorting concepts hierarchically...")

    # Create a copy to work with
    df_sorted = df.copy().reset_index(drop=True)

    # Extract parent IDs from parent_concept_urls
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, pd.Series, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            # Extract IDs from URLs like "/orgs/Regenstrief/sources/LOINC/concepts/ROOT/"
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    # Extract the concept ID from the URL
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:  # Expected format: orgs/Regenstrief/sources/LOINC/concepts/ID
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_sorted['parent_ids'] = df_sorted['parent_concept_urls'].apply(extract_parent_ids)

    # Topological sort
    sorted_concepts = []
    processed_ids = set()
    remaining_df = df_sorted.copy()

    iteration = 0
    max_iterations = len(df_sorted) + 10  # Safety limit

    while len(remaining_df) > 0 and iteration < max_iterations:
        iteration += 1

        # Find concepts whose parents are already processed (or have no parents)
        ready_concepts = []

        for idx, row in remaining_df.iterrows():
            parent_ids = row['parent_ids']

            # Concept is ready if it has no parents or all parents are already processed
            if not parent_ids or all(pid in processed_ids for pid in parent_ids):
                ready_concepts.append(idx)

        if not ready_concepts:
            # If no concepts are ready, we might have circular dependencies
            # Add remaining concepts in order of their dependency count
            print(f"Warning: Possible circular dependencies detected. Adding remaining {len(remaining_df)} concepts by dependency count.")
            dependency_counts = remaining_df['parent_ids'].apply(lambda x: len([p for p in x if p not in processed_ids]))
            ready_concepts = dependency_counts.sort_values().index.tolist()

        # Add ready concepts to sorted list
        for idx in ready_concepts:
            row = remaining_df.loc[idx]
            sorted_concepts.append(row)
            processed_ids.add(row['id'])

        # Remove processed concepts from remaining
        remaining_df = remaining_df.drop(ready_concepts)

        if iteration % 50 == 0:
            print(f"  Processed {len(sorted_concepts)}/{len(df_sorted)} concepts...")

    # Create the final sorted dataframe
    final_sorted_df = pd.DataFrame(sorted_concepts).reset_index(drop=True)

    # Drop the temporary parent_ids column
    final_sorted_df = final_sorted_df.drop('parent_ids', axis=1)

    print(f"Hierarchical sorting complete. Processed {len(final_sorted_df)} concepts in {iteration} iterations.")

    return final_sorted_df

# Debug code to identify circular dependencies - FIXED VERSION
def debug_circular_dependencies(df):
    """
    Analyzes the concepts dataframe to identify potential circular dependencies.
    """
    print("\n=== DEBUG: Analyzing Circular Dependencies ===")

    # Extract parent IDs from parent_concept_urls (FIXED VERSION)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, pd.Series, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_debug = df.copy()
    df_debug['parent_ids'] = df_debug['parent_concept_urls'].apply(extract_parent_ids)

    # Create a dictionary of concept -> parents mapping
    concept_parents = {}
    for _, row in df_debug.iterrows():
        concept_id = row['id']
        parent_ids = row['parent_ids']
        concept_parents[concept_id] = parent_ids

    # Find concepts that are involved in cycles
    def find_path_to_concept(start_concept, target_concept, visited=None):
        """Returns the path if target_concept is reachable from start_concept"""
        if visited is None:
            visited = set()

        if start_concept in visited:
            return None  # Already visited, potential cycle

        if start_concept == target_concept:
            return [start_concept]

        visited.add(start_concept)

        # Check all parents of the current concept
        parents = concept_parents.get(start_concept, [])
        for parent in parents:
            if parent in concept_parents:  # Only follow parents that exist as concepts
                path = find_path_to_concept(parent, target_concept, visited.copy())
                if path:
                    return [start_concept] + path

        return None

    # Check for cycles
    cycles_found = []
    concepts_in_cycles = set()

    for concept_id in concept_parents.keys():
        # Check if this concept can reach itself through its parents
        parents = concept_parents.get(concept_id, [])
        for parent in parents:
            if parent in concept_parents:
                cycle_path = find_path_to_concept(parent, concept_id)
                if cycle_path:
                    full_cycle = [concept_id] + cycle_path
                    cycles_found.append(full_cycle)
                    concepts_in_cycles.update(full_cycle)

    if cycles_found:
        print(f"🚨 Found {len(cycles_found)} circular dependency cycles:")
        for i, cycle in enumerate(cycles_found, 1):
            print(f"  Cycle {i}: {' → '.join(cycle)} → {cycle[0]}")

    # Identify concepts that might be causing the sorting issues
    problem_concepts = []
    all_concept_ids = set(concept_parents.keys())

    # Find concepts whose parents don't exist as concepts
    for concept_id, parent_ids in concept_parents.items():
        missing_parents = [p for p in parent_ids if p not in all_concept_ids]
        if missing_parents:
            problem_concepts.append({
                'concept_id': concept_id,
                'issue': 'missing_parents',
                'details': missing_parents
            })

    # Find concepts with complex parent relationships
    multi_parent_concepts = []
    for concept_id, parent_ids in concept_parents.items():
        if len(parent_ids) > 1:
            multi_parent_concepts.append({
                'concept_id': concept_id,
                'parent_count': len(parent_ids),
                'parents': parent_ids
            })

    print(f"\n📊 Dependency Analysis Summary:")
    print(f"   Total concepts: {len(concept_parents)}")
    print(f"   Concepts in cycles: {len(concepts_in_cycles)}")
    print(f"   Concepts with missing parents: {len(problem_concepts)}")
    print(f"   Concepts with multiple parents: {len(multi_parent_concepts)}")

    if problem_concepts:
        print(f"\n⚠️  Concepts with missing parents:")
        for problem in problem_concepts[:5]:  # Show first 5
            print(f"   {problem['concept_id']} → missing: {problem['details']}")
        if len(problem_concepts) > 5:
            print(f"   ... and {len(problem_concepts) - 5} more")

    if multi_parent_concepts:
        print(f"\n📋 Concepts with multiple parents:")
        for concept in multi_parent_concepts[:5]:  # Show first 5
            print(f"   {concept['concept_id']} → parents: {concept['parents']}")
        if len(multi_parent_concepts) > 5:
            print(f"   ... and {len(multi_parent_concepts) - 5} more")

    # Show the specific concepts that are likely causing the sorting warning
    if concepts_in_cycles:
        print(f"\n🔍 Concepts likely causing the sorting warning:")
        cycle_concept_details = df_debug[df_debug['id'].isin(concepts_in_cycles)][['id', 'concept_class', 'parent_ids']]
        print(cycle_concept_details.to_string(index=False))

    print("=== End Debug Analysis ===\n")

    return concepts_in_cycles, problem_concepts, multi_parent_concepts

# Run the debug analysis
debug_results = debug_circular_dependencies(concepts_with_containers_df)

# Fix missing parent references by reassigning to containers
def fix_missing_parent_references(df):
    """
    Identifies concepts with missing parent references and reassigns them to appropriate containers.
    """
    print("\n=== FIXING: Missing Parent References ===")

    # Get all concept IDs that exist in our dataset
    existing_concept_ids = set(df['id'].values)

    # Container mapping
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER',
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }

    # Extract parent IDs function (same logic as above)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []

        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, pd.Series, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []

        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []

        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []

    df_fixed = df.copy()
    concepts_fixed = []

    for idx, row in df_fixed.iterrows():
            concept_id = row['id']
            parent_urls = row['parent_concept_urls']
            concept_class = row['concept_class']

            # Use extract_parent_ids to safely handle all parent_urls formats
            parent_ids = extract_parent_ids(parent_urls)

            if parent_ids:  # Only proceed if we have parent IDs to check
                missing_parents = [pid for pid in parent_ids if pid not in existing_concept_ids]

                if missing_parents:
                    # This concept has missing parent references
                    # Reassign to appropriate container
                    container_id = concept_class_to_container.get(concept_class, 'ROOT')
                    new_parent_url = f"{CONCEPT_URL_PREFIX}{container_id}/"

                    # Update the parent_concept_urls
                    df_fixed.at[idx, 'parent_concept_urls'] = [new_parent_url]

                    concepts_fixed.append({
                        'concept_id': concept_id,
                        'concept_class': concept_class,
                        'missing_parents': missing_parents,
                        'new_parent': container_id
                    })

    print(f"✅ Fixed {len(concepts_fixed)} concepts with missing parent references")
    print("=== End Fix ===\n")

    return df_fixed, concepts_fixed

# Apply the fix
concepts_with_containers_df_fixed, fixed_concepts = fix_missing_parent_references(concepts_with_containers_df)

# Update the variable name for clarity
concepts_with_containers_df = concepts_with_containers_df_fixed

# Apply hierarchical sorting
final_concepts_with_hierarchy_df = sort_concepts_hierarchically(concepts_with_containers_df)

# 6. Update the global dataframe with the final result
merged_loinc_cui_df_for_output = final_concepts_with_hierarchy_df.copy()

print("=== Container Concepts Creation Complete ===")

### Output Generation - Concepts and Mappings

In [ ]:
# Output - Generate OCL Concepts and Mappings JSON

output_folder_path = 'output'
os.makedirs(output_folder_path, exist_ok=True)

CHUNK_SIZE = 15000

def is_valid_value(value):
    """
    Checks if a value is not NaN, None, an empty list/dict, or an empty string.
    Handles non-scalar values (like lists and pandas Series) safely.
    """
    if value is None:
        return False

    # Handle empty string specifically
    if isinstance(value, str):
         return value.strip() != "" # Keep any non-empty string after stripping whitespace

    # Handle scalar values first
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        return not pd.isna(value)

    # Handle non-scalar values (lists, Series, etc.)
    # Consider them valid only if they contain at least one non-NaN value
    if isinstance(value, pd.Series):
        return not value.dropna().empty
    elif isinstance(value, (list, np.ndarray)):
        # Check for empty list/array or list/array of NaNs
        return any(pd.notna(v) for v in value)

    # For other types, check if they are "empty"
    if hasattr(value, '__len__') and len(value) == 0:
        return False

    return True


def safe_na_to_none(value):
    """
    Safely converts NaN values to None, handling both scalar and array-like values.
    This prevents the ValueError: The truth value of an array with more than one element is ambiguous.
    FIXED: Preserves actual CUI values (like "C0000097") that might look like they need conversion.
    """
    # Handle None explicitly
    if value is None:
        return None

    # FIXED: For CUI fields, preserve string values even if they might trigger false positives
    if isinstance(value, str) and value.strip():
        # Don't convert valid CUI strings or other meaningful strings
        return value

    # Handle scalar values (numbers, single values)
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        try:
            return None if pd.isna(value) else value
        except (TypeError, ValueError):
            # If pd.isna() fails for any reason, return the original value
            return value

    # Handle array-like values (lists, arrays, Series)
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return None
        # Check if all elements are NaN
        try:
            if all(pd.isna(v) for v in value):
                return None
            else:
                return value
        except (TypeError, ValueError):
            return value

    elif isinstance(value, pd.Series):
        if value.empty or value.dropna().empty:
            return None
        else:
            return value

    elif isinstance(value, np.ndarray):
        if value.size == 0:
            return None
        try:
            if np.all(pd.isna(value)):
                return None
            else:
                return value
        except (TypeError, ValueError):
            return value

    return value


def clean_concept_for_json(concept):
    """
    Cleans a concept dictionary for JSON output by removing None and empty collections
    and ensuring correct structure for names, descriptions, and extras.
    """
    cleaned_concept = {}
    for key, value in concept.items():
        # Apply safe_na_to_none to all values first
        cleaned_value = safe_na_to_none(value)

        # Check if the cleaned value is valid (not None, empty string, or empty collection)
        if is_valid_value(cleaned_value):
            cleaned_concept[key] = cleaned_value

    # Ensure 'names', 'descriptions', and 'extras' are dictionaries/lists if present and not empty
    for list_key in ['names', 'descriptions']:
        if list_key in cleaned_concept and not cleaned_concept[list_key]:
            del cleaned_concept[list_key] # Remove empty list

    if 'extras' in cleaned_concept and not cleaned_concept['extras']:
         del cleaned_concept['extras'] # Remove empty dict

    return cleaned_concept

# --- CONCEPTS OUTPUT ---
print("=== Generating Concepts JSON Output ===")

concepts_output_filename = 'concepts.json'
concepts_output_path = os.path.join(output_folder_path, concepts_output_filename)

print(f"Writing concepts to {concepts_output_path}...")

# Define the list of concepts that should have ROOT as their parent
root_exceptions = {
    'LP29693-6',
    'LP29695-1',
    'LP29696-9',
    'LP7787-7'
}
root_parent_url = f"{CONCEPT_URL_PREFIX}{'ROOT'}/"


# Process and clean concepts from the final dataframe
concepts_data = []
for idx, row in merged_loinc_cui_df_for_output.iterrows():
    row_dict = row.to_dict()

    # *** NEW LOGIC: Explicitly set parent_concept_urls for root_exceptions here ***
    concept_id = row_dict.get('id')
    if concept_id in root_exceptions:
        row_dict['parent_concept_urls'] = [root_parent_url]
        print(f"Explicitly setting parent for {concept_id} to {root_parent_url} for JSON output.") # Optional debug print


    # Logic to dynamically process all 'names' attributes with locale_preferred flag
    names_list = []
    # Iterate over a copy of keys because we'll modify the dict during iteration
    for key, value in list(row_dict.items()):
        if key.startswith('names.'):
            if is_valid_value(value):
                # Use regex to extract name_type and locale
                match = re.match(r'names\.([^.]+)\.([^\[]+)\[\d+\]', key)
                if match:
                    name_type, locale = match.groups()
                    # Determine locale_preferred based on name_type
                    # For LOINC, Long-Common is preferred. For Parts, Fully-Specified.
                    # Need to handle this based on concept_class or hardcode for LOINC
                    # Assuming Long-Common is preferred for simplicity here, adjust if needed
                    locale_preferred = (name_type == 'Long-Common') # Default for LOINC

                    # Check concept_class for specific rules
                    concept_class = row_dict.get('concept_class')
                    if concept_class == 'LOINC Part' and name_type == 'Fully-Specified':
                         locale_preferred = True
                    elif concept_class in ['Answer List', 'LOINC Answer'] and name_type == 'Display':
                         locale_preferred = True
                    elif concept_class in ['Root', 'Container'] and name_type == 'Fully-Specified':
                         locale_preferred = True
                    else:
                         locale_preferred = False # Default to False if not explicitly preferred

                    names_list.append({'name': value, 'name_type': name_type, 'locale': locale, 'locale_preferred': locale_preferred})
            del row_dict[key] # Remove the original 'names.xxx' key

    if names_list:
        row_dict['names'] = names_list

    # Handle description
    description = row_dict.pop('description', None)
    if is_valid_value(description):
        row_dict['descriptions'] = [{'description': description, 'locale': 'en-GB', 'description_type': 'Full'}]
    else:
        row_dict['descriptions'] = [] # Ensure descriptions key exists, even if empty

    # Handle extras
    extras_dict = {}
    for key, value in list(row_dict.items()):
        if key.startswith('extras.'):
            # Apply safe_na_to_none before checking is_valid_value for extras
            cleaned_extra_value = safe_na_to_none(value)
            if is_valid_value(cleaned_extra_value):
                new_key = key[len('extras.'):]
                extras_dict[new_key] = cleaned_extra_value
            del row_dict[key]

    if extras_dict:
        row_dict['extras'] = extras_dict
    else:
        row_dict['extras'] = {} # Ensure extras key exists, even if empty


    # Handle parent_concept_urls - this logic now runs *after* potential override for root_exceptions
    parent_urls = row_dict.pop('parent_concept_urls', None)
    if is_valid_value(parent_urls):
        # Ensure it's always a list of strings
        if isinstance(parent_urls, str):
            row_dict['parent_concept_urls'] = [parent_urls]
        elif isinstance(parent_urls, (list, tuple, np.ndarray)):
             # Filter out any non-string or empty string values from the list
             valid_urls = [url for url in parent_urls if isinstance(url, str) and url.strip()]
             if valid_urls:
                 row_dict['parent_concept_urls'] = valid_urls
             else:
                 # If the list was not empty but contained only invalid URLs, treat as None
                 print(f"Skipping {concept_id} - parent is {parent_urls} for JSON output.") # Optional debug print
                 pass # Do nothing, parent_concept_urls will not be added
        else:
            # If it's some other non-valid type, treat as None
            print(f"Skipping {concept_id} --- parent is {parent_urls} for JSON output.") # Optional debug print
            pass # Do nothing

    # Final cleanup of the concept dictionary
    cleaned_record = clean_concept_for_json(row_dict)

    concepts_data.append(cleaned_record)

# Write concepts to JSON Lines file
with open(concepts_output_path, 'w', encoding='utf-8') as f:
    for record in concepts_data:
        json.dump(record, f, ensure_ascii=False)
        f.write('\n')

print(f"Successfully wrote {len(concepts_data)} concepts to {concepts_output_path}")


# --- MAPPINGS OUTPUT ---
print("\n=== Generating Mappings JSON Output ===")

mappings_output_filename = 'mappings.json'
mappings_output_path = os.path.join(output_folder_path, mappings_output_filename)

print(f"Writing mappings to {mappings_output_path}...")

# Process and clean mappings from the all_mappings list
mappings_data = []
for mapping in all_mappings:
    mapping_dict = mapping.copy() # Work with a copy

    # Handle extras for mappings
    extras_dict = {}
    # Iterate over a copy of keys
    for key, value in list(mapping_dict.items()):
        if key.startswith('extras.'):
            # Apply safe_na_to_none before checking is_valid_value for extras
            cleaned_extra_value = safe_na_to_none(value)
            if is_valid_value(cleaned_extra_value):
                new_key = key[len('extras.'):]
                extras_dict[new_key] = cleaned_extra_value
            del mapping_dict[key] # Remove the original 'extras.xxx' key

    if extras_dict:
        mapping_dict['extras'] = extras_dict
    else:
        mapping_dict['extras'] = {} # Ensure extras key exists, even if empty

    # Final cleanup of the mapping dictionary
    cleaned_record = clean_concept_for_json(mapping_dict) # Reuse the cleaning function

    mappings_data.append(cleaned_record)


# Write mappings to JSON Lines file
if mappings_data:
    with open(mappings_output_path, 'w', encoding='utf-8') as f:
        for record in mappings_data:
            json.dump(record, f, ensure_ascii=False)
            f.write('\n')

    print(f"Successfully wrote {len(mappings_data)} mappings to {mappings_output_path}")
else:
    print("No mappings data to write.")

print("\n✅ Output generation complete!")
print(f"   Output files saved in the '{output_folder}' directory.")

In [ ]:
# Optional - Hierarchy-Only Output - Generate parent-child concept URL relationships

hierarchy_only = True  # Variable to enable hierarchy-only mode

def build_concept_url(concept_id, org=global_organization, source=global_source):
    """
    Builds a concept URL from concept ID following OCL URL pattern.

    Args:
        concept_id: The concept ID
        org: Organization name (default: 'Regenstrief')
        source: Source name (default: 'LOINC')

    Returns:
        str: Full concept URL
    """
    return f"/orgs/{org}/sources/{source}/concepts/{concept_id}/"

def build_hierarchy_mapping(data_df):
    """
    Builds a hierarchy mapping dictionary where keys are parent concept URLs
    and values are lists of child concept URLs.

    Args:
        data_df: DataFrame containing concept data with 'id' and 'parent_concept_urls' columns

    Returns:
        dict: Parent URL -> [Child URLs] mapping
    """
    print("Building hierarchy mapping...")

    hierarchy_map = {}
    total_concepts = len(data_df)
    concepts_with_parents = 0

    for idx, row in data_df.iterrows():
        # if (idx + 1) % 1000 == 0 or (idx + 1) == total_concepts:
        # print(f"   Processed {idx + 1}/{total_concepts} concepts for hierarchy.")

        concept_id = row.get('id')
        if not concept_id:
            continue

        # Build the current concept's URL
        child_url = build_concept_url(concept_id)

        # Get parent URLs
        parent_urls = row.get('parent_concept_urls', [])

        # Handle case where parent_urls might be NaN or None
        if parent_urls is None or (hasattr(parent_urls, '__len__') and len(parent_urls) == 0):
            continue

        # Ensure parent_urls is a list
        if not isinstance(parent_urls, (list, tuple)):
            try:
                if pd.isna(parent_urls):
                    continue
            except (TypeError, ValueError):
                pass
            # If it's a single URL string, convert to list
            if isinstance(parent_urls, str):
                parent_urls = [parent_urls]
            else:
                continue

        # Add this concept as a child to each of its parents
        for parent_url in parent_urls:
            if parent_url and isinstance(parent_url, str) and parent_url.strip():
                parent_url = parent_url.strip()

                # Initialize parent entry if it doesn't exist
                if parent_url not in hierarchy_map:
                    hierarchy_map[parent_url] = []

                # Add child URL if not already present
                if child_url not in hierarchy_map[parent_url]:
                    hierarchy_map[parent_url].append(child_url)

        if parent_urls:
            concepts_with_parents += 1

    # --- ADDED CODE START ---
    # Manually add the specified children to the ROOT concept.
    root_url = f"/orgs/{global_organization}/sources/{global_source}/concepts/ROOT/"
    new_root_children_ids = [
        "LP29693-6",
        "LP29695-1",
        "LP29696-9",
        "LP7787-7"
    ]

    # Create the full OCL URLs for the new children
    new_root_children_urls = [build_concept_url(child_id) for child_id in new_root_children_ids]

    # Add the new children to the hierarchy map
    if root_url not in hierarchy_map:
        hierarchy_map[root_url] = []

    for child_url in new_root_children_urls:
        if child_url not in hierarchy_map[root_url]:
            hierarchy_map[root_url].append(child_url)

    # --- ADDED CODE END ---

    print(f"Hierarchy mapping complete:")
    print(f"  - Total concepts processed: {total_concepts:,}")
    print(f"  - Concepts with parents: {concepts_with_parents:,}")
    print(f"  - Parent concepts in hierarchy: {len(hierarchy_map):,}")

    # Show some statistics
    child_counts = [len(children) for children in hierarchy_map.values()]
    if child_counts:
        print(f"  - Average children per parent: {sum(child_counts) / len(child_counts):.1f}")
        print(f"  - Max children for a parent: {max(child_counts):,}")
        print(f"  - Total parent-child relationships: {sum(child_counts):,}")

    return hierarchy_map

def write_hierarchy_json(hierarchy_map, output_filename='hierarchy_only.json'):
    """
    Writes the hierarchy mapping to a JSON file.

    Args:
        hierarchy_map: Dictionary with parent URLs as keys and child URL lists as values
        output_filename: Name of the output file
    """
    output_file_path = os.path.join(output_folder, output_filename)

    print(f"Writing hierarchy-only JSON to {output_file_path}...")

    # Sort the hierarchy map by parent URL for consistent output
    sorted_hierarchy = dict(sorted(hierarchy_map.items()))

    # Also sort the child lists for each parent
    for parent_url in sorted_hierarchy:
        sorted_hierarchy[parent_url] = sorted(sorted_hierarchy[parent_url])

    # Write to JSON file with proper encoding
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(sorted_hierarchy, f, ensure_ascii=False, indent=2)

    print(f"Hierarchy-only JSON written successfully!")
    print(f"File size: {os.path.getsize(output_file_path):,} bytes")

# Execute hierarchy-only processing if enabled
if hierarchy_only:
    print("=" * 60)
    print("HIERARCHY-ONLY MODE ENABLED")
    print("=" * 60)

    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Build the hierarchy mapping
    hierarchy_mapping = build_hierarchy_mapping(merged_loinc_cui_df_for_output)

    # Write to JSON file
    write_hierarchy_json(hierarchy_mapping, 'hierarchy_only.json')

    # Show a sample of the output
    # print("\nSample hierarchy relationships:")
    # sample_count = 0
    # for parent_url, child_urls in hierarchy_mapping.items():
    #     if sample_count >= 3:  # Show first 3 examples
    #         break
    #     print(f"  {parent_url}: {child_urls}")
    #     sample_count += 1

    # if len(hierarchy_mapping) > 3:
    #     print(f"  ... and {len(hierarchy_mapping) - 3} more parent-child relationships")

    print(f"\nHierarchy-only processing complete!")
    print(f"Output file: {os.path.join(output_folder, 'hierarchy_only.json')}")

else:
    print("Hierarchy-only mode is disabled. Set hierarchy_only = True to enable.")

# Now What?

1. Review the outputted files to ensure there aren't any unexpected attribute values like 'NaN', all mappings have a to and from URL, etc.
2. Make sure the LOINC source is available on your environment, which may involve loading the LOINC-Source-Import JSON line.
3. Import only the "Root" concept line to the LOINC source.
4. Use a "PUT" command to update the LOINC source with the hierarchy_root_URL attribute, making sure it appears on the source API GET call.
5. Begin importing files using OCL's Bulk Import interface. Do NOT check the Hierarchy checkbox (this will be applied later).
6. Use the Admin API call to apply the hierarchy using the hierarchy_only file.
7. Check results: concept quality, mapping quality, counts of each, etc.